# Hybrid 1D-CNN Autoencoder–CNN Classifier

Same 3-head design as deep_multitask_mlp (reconstruction + anomaly + 7-class classifier), but the shared encoder is a small plain (non-dilated) Conv1D stack — a CNN-based counterpart to both deep_multitask_mlp (Dense) and tcn_hybrid (dilated Conv1D).

Reconstruction is trained on the BROADER safe set (normal + motor_off + transition_on + transition_off) — same reasoning as deep_multitask_mlp: motor_off/transitions are legitimate states, not anomalies.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn joblib openpyxl matplotlib tensorflow tf2onnx onnx

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedShuffleSplit,
)
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_PATH = (
    r"../../data"
    r"\Motor_data_model"
)

SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s",
    "PWM",
    "Irms",
    "Vrms",
    "Current_Variance",
    "Current_Crest_Factor",
    "Vib_RMS",
    "Vib_Min",
    "Vib_Q1",
    "Vib_Median",
    "Vib_Q3",
    "Vib_Max",
    "Vib_Variance",
    "Vib_Skewness",
]

FINAL_LABELS = [
    "motor_off",
    "turning_off",
    "turning_on",
    "normal",
    "electrical_fault",
    "mechanical_fault",
    "thermal_fault",
]

LABEL_TO_ID = {
    label: index
    for index, label in enumerate(FINAL_LABELS)
}

SAFE_LABELS = [
    "motor_off",
    "turning_off",
    "turning_on",
    "normal",
]

FAULT_LABELS = [
    "electrical_fault",
    "mechanical_fault",
    "thermal_fault",
]

SAFE_CLASS_IDS = np.asarray(
    [LABEL_TO_ID[label] for label in SAFE_LABELS],
    dtype=np.int64,
)

FAULT_CLASS_IDS = np.asarray(
    [LABEL_TO_ID[label] for label in FAULT_LABELS],
    dtype=np.int64,
)

N_FEATURES = len(FEATURE_COLUMNS)
N_CLASSES = len(FINAL_LABELS)

WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
BATCH_SIZE = 128
MAX_EPOCHS = 180

LEARNING_RATE = 1e-3

# CHANGED: RECON_LOSS_WEIGHT raised from 0.20 -> 1.00. Now that the
# reconstruction encoder is fully isolated (CELL 8 dual-encoder fix),
# it no longer competes with anomaly/class gradients for a shared
# trunk, so a low weight is no longer needed to protect those heads
# — and a low weight was only slowing the recon branch's convergence.
RECON_LOSS_WEIGHT = 1.00
ANOMALY_LOSS_WEIGHT = 0.80
CLASS_LOSS_WEIGHT = 1.00

# Compact but not excessively weak.
ENCODER_FILTERS_1 = 24
ENCODER_FILTERS_2 = 12
LATENT_FILTERS = 8

# Operating-context branch
CONTEXT_DENSE_UNITS = 12

CLASS_DENSE_UNITS = 16
ANOMALY_DENSE_UNITS = 12

DROPOUT_RATE = 0.10
L2_REGULARIZATION = 1e-5

# STM32-safe convolution settings: dilation is ALWAYS 1 so the
# exported graph never contains SpaceToBatchND/BatchToSpaceND.
# Kernel size is 5 (instead of 3) to recover a comparable
# receptive field across the 30-step window.
CONV_KERNEL_SIZE = 3
CONV_DILATION_RATE = 1

CLASS_CONF_THRESHOLD = 0.65
ANOMALY_PROB_THRESHOLD = 0.65
RECON_QUANTILE = 0.990

# ADDED: reconstruction severity tiers (matches DANN notebook).
# Pure post-processing on reconstruction_mse / local_baseline —
# no retraining involved.
SEVERITY_MILD_MULTIPLIER = 2.0
SEVERITY_MODERATE_MULTIPLIER = 4.0
SEVERITY_SEVERE_MULTIPLIER = 8.0

# Reconstruction thresholds are calculated only from
# held-out normal-running windows in each PWM range.
#
# NOTE: the upper edge of 1.01 already acts as a soft ceiling for
# PWM values slightly above 100% caused by ulong-scaling logging
# artifacts at full duty cycle. The real fix is clipping PWM at the
# source (see CELL 4) so those rows don't silently contaminate the
# "75_100" bin's calibration — this bin-edge value stays as a safety
# margin, not a substitute for the source-level fix.
PWM_BIN_EDGES = np.asarray([
    0.00,
    0.20,
    0.35,
    0.45,
    0.55,
    0.65,
    0.75,
    1.01,
], dtype=np.float32)

PWM_BIN_NAMES = [
    "0_20",
    "20_35",
    "35_45",
    "45_55",
    "55_65",
    "65_75",
    "75_100",
]

# Reconstruction reference:
# - motor_off / turning_off / turning_on use state-specific thresholds.
# - steady normal and fault windows use a normal-reference threshold
#   conditioned on both PWM and an Irms-based load proxy.
LOAD_PROXY_FEATURE = "Irms"
LOAD_SPLIT_QUANTILE = 0.50
MIN_THRESHOLD_WINDOWS = 30

LOAD_STATE_NAMES = [
    "low_load",
    "high_load",
]

OUTPUT_DIR = "conv1d_ae_cnn_3head_safe_output"

CV_DIR = os.path.join(
    OUTPUT_DIR,
    "cross_validation",
)

FINAL_DIR = os.path.join(
    OUTPUT_DIR,
    "final_model",
)

MANUAL_TEST_DIR = os.path.join(
    OUTPUT_DIR,
    "manual_test_results",
)

for folder in [
    CV_DIR,
    FINAL_DIR,
    MANUAL_TEST_DIR,
]:
    os.makedirs(
        folder,
        exist_ok=True,
    )

print("Model: 3-Head Conv1D Hybrid (STM32-SAFE, dual-encoder, LayerNorm)")
print("Input:", WINDOW_LEN, "x", N_FEATURES)
print("Classes:", FINAL_LABELS)
print("Safe classes:", SAFE_LABELS)
print("Fault classes:", FAULT_LABELS)

In [ ]:
# ==========================================================
# CELL 3 — deterministic file-to-class mapping
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def label_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if any(term in path_text for term in [
        "motor_being_off",
        "turning_off",
        "transition_off",
        "off_spikes",
    ]):
        return "turning_off"

    if any(term in path_text for term in [
        "motor_being_on",
        "turning_on",
        "transition_on",
        "on_spikes",
    ]):
        return "turning_on"

    if "/motor_off/" in path_text or "motor_off" in stem:
        return "motor_off"

    if any(term in path_text for term in [
        "electrical fault",
        "electrical_fault",
        "abnormal_current",
    ]):
        return "electrical_fault"

    if any(term in path_text for term in [
        "mechanical fault",
        "mechanical_fault",
        "abnormal_vibration",
    ]):
        return "mechanical_fault"

    if any(term in path_text for term in [
        "thermal fault",
        "thermal_fault",
        "abnormal_temp",
    ]):
        return "thermal_fault"

    if (
        "/normal/" in path_text
        or stem.startswith("normal_")
        or stem == "normal"
    ):
        return "normal"

    return None


def anomaly_target_from_class_id(class_id):
    return int(
        class_id in FAULT_CLASS_IDS
    )

# ----------------------------------------------------------
# CHANGED: reconstruction reference = NORMAL WITH LOAD only
# (same rule as the DANN notebook). Without-load normal,
# motor_off and transitions are NOT used to train the AE.
# ----------------------------------------------------------
def is_normal_with_load_file(path, label):
    if label != "normal":
        return False

    path_text = normalized_path(path)

    # Check without-load first because it also contains the word "load".
    without_load_terms = [
        "without_load",
        "without load",
        "no_load",
        "no load",
        "unloaded",
        "normal_noload",
        "normal_no_load",
    ]

    with_load_terms = [
        "with_load",
        "with load",
        "loaded",
        "normal_load",
    ]

    if any(term in path_text for term in without_load_terms):
        return False

    if any(term in path_text for term in with_load_terms):
        return True

    raise ValueError(
        "Cannot identify loaded/unloaded normal file: "
        f"{path}"
    )


In [ ]:
# ==========================================================
# CELL 4 — discover and read files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(
        os.path.join(base_path, "**", "*.xlsx"),
        recursive=True,
    )

    files += glob.glob(
        os.path.join(base_path, "**", "*.csv"),
        recursive=True,
    )

    return sorted(set(files))


def read_one(path):

    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(
            f"Unsupported file type: {path}"
        )

    missing_columns = [
        column
        for column in FEATURE_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in {path.name}: "
            f"{missing_columns}"
        )

    clean = df[FEATURE_COLUMNS].apply(
        pd.to_numeric,
        errors="coerce",
    )

    clean = clean / SCALE_DIVISOR

    # ADDED: PWM duty cycle is physically bounded to [0, 1]. Values
    # recorded slightly above 1.0 are ulong-scaling logging artifacts
    # at the 100% boundary, not a real operating condition. Without
    # this clip they silently contaminate the top PWM bin's
    # reconstruction-threshold calibration.
    if "PWM" in clean.columns:
        overshoot_count = int((clean["PWM"] > 1.0).sum())
        if overshoot_count > 0:
            print(f"  Clipping {overshoot_count} PWM rows > 1.0 in {path.name}")
        clean["PWM"] = clean["PWM"].clip(upper=1.0)

    clean = clean.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(
            f"{path.name} has only {len(clean)} valid rows."
        )

    return clean

In [ ]:
# ==========================================================
# CELL 5 — load labeled sessions
# ==========================================================
files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: {BASE_PATH}"
    )

sessions = []

for session_id, file_path in enumerate(files):
    label = label_for(file_path)

    if label is None:
        print(
            "SKIP unrecognized:",
            os.path.relpath(file_path, BASE_PATH),
        )
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(file_path, BASE_PATH),
            "|",
            error,
        )
        continue

    class_id = LABEL_TO_ID[label]

    sessions.append({
        "session_id": session_id,
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "label": label,
        "class_id": class_id,
        "anomaly": anomaly_target_from_class_id(class_id),
        "reconstruction_reference": int(is_normal_with_load_file(file_path, label)),  # CHANGED
        "rows": len(df),
    })

if not sessions:
    raise RuntimeError(
        "No labeled sessions were loaded."
    )

class_counts = pd.Series([
    session["label"]
    for session in sessions
]).value_counts().reindex(
    FINAL_LABELS,
    fill_value=0,
)

print("\nFILES PER CLASS")
print(class_counts)

missing_classes = [
    label
    for label in FINAL_LABELS
    if class_counts[label] == 0
]

if missing_classes:
    raise RuntimeError(
        f"No files found for classes: {missing_classes}"
    )


# ----------------------------------------------------------
# CHANGED: session_id -> 1 if normal-with-load (AE reference)
# ----------------------------------------------------------
RECON_REFERENCE_BY_SESSION = {
    session["session_id"]: session["reconstruction_reference"]
    for session in sessions
}

print("\nNORMAL WITH LOAD — USED TO TRAIN THE RECONSTRUCTION AE")
for session in sessions:
    if session["reconstruction_reference"] == 1:
        print("  ", session["file"])

print("\nNORMAL WITHOUT LOAD — NOT USED FOR AE TRAINING")
for session in sessions:
    if session["label"] == "normal" and session["reconstruction_reference"] == 0:
        print("  ", session["file"])

if not any(RECON_REFERENCE_BY_SESSION.values()):
    raise RuntimeError("No normal-with-load files were identified.")


In [ ]:
# ==========================================================
# CELL 6 — create 30 x 14 windows
# ==========================================================
def make_windows_from_array(X_raw):
    starts = np.arange(
        0,
        len(X_raw) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_raw[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return windows, starts


def build_dataset_from_sessions(selected_sessions):
    all_windows = []
    all_class_labels = []
    all_anomaly_labels = []
    all_files = []
    all_starts = []
    all_groups = []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(
            session["X"]
        )

        if windows is None:
            continue

        number_of_windows = len(windows)

        all_windows.append(windows)

        all_class_labels.extend(
            [session["class_id"]] * number_of_windows
        )

        all_anomaly_labels.extend(
            [session["anomaly"]] * number_of_windows
        )

        all_files.extend(
            [session["file"]] * number_of_windows
        )

        all_starts.extend(
            starts.tolist()
        )

        all_groups.extend(
            [session["session_id"]] * number_of_windows
        )

    if not all_windows:
        raise RuntimeError(
            "No windows were created."
        )

    return (
        np.concatenate(
            all_windows,
            axis=0,
        ).astype(np.float32),
        np.asarray(
            all_class_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_anomaly_labels,
            dtype=np.float32,
        ),
        np.asarray(
            all_files,
            dtype=object,
        ),
        np.asarray(
            all_starts,
            dtype=np.int64,
        ),
        np.asarray(
            all_groups,
            dtype=np.int64,
        ),
    )

In [ ]:
# ==========================================================
# CELL 7 — normalization utilities
# ==========================================================
def calculate_normalization_statistics(X_train):
    flattened_rows = X_train.reshape(
        -1,
        N_FEATURES,
    )

    mean = np.mean(
        flattened_rows,
        axis=0,
    ).astype(np.float32)

    variance = np.var(
        flattened_rows,
        axis=0,
    ).astype(np.float32)

    variance = np.maximum(
        variance,
        1e-12,
    )

    return mean, variance


def normalize_windows_numpy(
    X,
    mean,
    variance,
):
    return (
        (
            X
            - mean.reshape(
                1,
                1,
                N_FEATURES,
            )
        )
        / np.sqrt(
            variance.reshape(
                1,
                1,
                N_FEATURES,
            )
            + 1e-7
        )
    ).astype(np.float32)

In [ ]:
# ==========================================================
# CELL 8 — operating-context-aware temporal CNN (STM32-SAFE)
# DUAL-ENCODER VERSION.
#
# WHY THIS CHANGE:
# Previously `latent` (shared temporal_block_1 -> temporal_block_2
# -> latent_temporal_block) fed BOTH the reconstruction decoder AND,
# via pooling, the anomaly/class heads. anomaly_output/class_output
# are trained with full weight on every window, while reconstruction
# trains only on safe windows — so the classifier pulls the shared
# latent toward fault-discriminative structure, which then gets
# decoded, corrupting the reconstruction error as an off-manifold
# signal. This mirrors the fix already applied to the MLP (DANN)
# model, so both notebooks are now structurally aligned for a fair
# comparison.
#
# FIX: two fully independent Conv1D encoder stacks, same three named
# outputs (nothing downstream needs to change):
#   recon_encoder  -> recon_latent  -> decoder -> reconstruction_output
#       (gradients ONLY from window_mse_loss on safe windows)
#   shared_encoder -> shared_latent -> pooled -> fused with context
#       -> anomaly_output / class_output
#       (gradients from anomaly + class losses on all windows)
#
# The operating-context branch (avg/max pooling directly on the
# normalized input) was already isolated from reconstruction and is
# kept exactly as-is.
# ==========================================================
def window_mse_loss(y_true, y_pred):
    """Return one reconstruction MSE value per complete window."""
    squared_error = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_error, axis=[1, 2])


def temporal_residual_block(
    inputs,
    filters,
    block_name,
    kernel_size=CONV_KERNEL_SIZE,
):
    """
    Residual Conv1D block — STM32-safe variant.
    dilation_rate is always 1 (plain Conv1D -> Conv2D lowering, no
    SpaceToBatchND/BatchToSpaceND). LayerNormalization replaces
    BatchNormalization.
    """
    shortcut = inputs

    x = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        dilation_rate=CONV_DILATION_RATE,
        padding="same",
        use_bias=False,
        kernel_regularizer=tf.keras.regularizers.l2(L2_REGULARIZATION),
        name=f"{block_name}_conv1",
    )(inputs)

    x = tf.keras.layers.LayerNormalization(name=f"{block_name}_ln1")(x)
    x = tf.keras.layers.Activation("relu", name=f"{block_name}_relu1")(x)
    x = tf.keras.layers.Dropout(DROPOUT_RATE, name=f"{block_name}_dropout")(x)

    x = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        dilation_rate=CONV_DILATION_RATE,
        padding="same",
        use_bias=False,
        kernel_regularizer=tf.keras.regularizers.l2(L2_REGULARIZATION),
        name=f"{block_name}_conv2",
    )(x)

    x = tf.keras.layers.LayerNormalization(name=f"{block_name}_ln2")(x)

    if int(shortcut.shape[-1]) != filters:
        shortcut = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same",
            use_bias=False,
            name=f"{block_name}_shortcut",
        )(shortcut)

    x = tf.keras.layers.Add(name=f"{block_name}_add")([shortcut, x])
    return tf.keras.layers.Activation("relu", name=f"{block_name}_output")(x)


def build_hybrid_model(
    normalization_mean,
    normalization_variance,
):
    input_layer = tf.keras.Input(
        shape=(WINDOW_LEN, N_FEATURES),
        name="motor_window",
    )

    normalizer = tf.keras.layers.Normalization(
        axis=-1,
        mean=normalization_mean,
        variance=normalization_variance,
        name="input_normalization",
    )
    normalized = normalizer(input_layer)

    # ======================================================
    # OPERATING-CONTEXT BRANCH (unchanged — already isolated
    # from reconstruction; feeds only anomaly/class heads)
    # ======================================================
    context_average = tf.keras.layers.GlobalAveragePooling1D(
        name="context_average",
    )(normalized)

    context_maximum = tf.keras.layers.GlobalMaxPooling1D(
        name="context_maximum",
    )(normalized)

    context = tf.keras.layers.Concatenate(name="operating_context_raw")(
        [context_average, context_maximum]
    )

    context = tf.keras.layers.Dense(
        CONTEXT_DENSE_UNITS,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(L2_REGULARIZATION),
        name="operating_context_dense",
    )(context)

    context = tf.keras.layers.Dropout(
        DROPOUT_RATE, name="operating_context_dropout",
    )(context)

    # ======================================================
    # BRANCH A — DEDICATED RECONSTRUCTION ENCODER (isolated).
    # Only reconstruction_output's loss ever touches these layers.
    # ======================================================
    recon_stem = tf.keras.layers.Conv1D(
        filters=ENCODER_FILTERS_1,
        kernel_size=1,
        padding="same",
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(L2_REGULARIZATION),
        name="recon_feature_interaction_conv",
    )(normalized)

    recon_stage_1 = temporal_residual_block(
        inputs=recon_stem,
        filters=ENCODER_FILTERS_1,
        block_name="recon_temporal_block_1",
    )

    recon_stage_2 = temporal_residual_block(
        inputs=recon_stage_1,
        filters=ENCODER_FILTERS_2,
        block_name="recon_temporal_block_2",
    )

    recon_latent = temporal_residual_block(
        inputs=recon_stage_2,
        filters=LATENT_FILTERS,
        block_name="recon_latent_temporal_block",
    )

    reconstruction_features = tf.keras.layers.Conv1D(
        filters=ENCODER_FILTERS_1,
        kernel_size=CONV_KERNEL_SIZE,
        padding="same",
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(L2_REGULARIZATION),
        name="decoder_conv",
    )(recon_latent)

    reconstruction_output = tf.keras.layers.Conv1D(
        filters=N_FEATURES,
        kernel_size=1,
        padding="same",
        activation="linear",
        name="reconstruction_output",
    )(reconstruction_features)

    # ======================================================
    # BRANCH B — SHARED DISCRIMINATIVE ENCODER (anomaly + class
    # only). Completely separate weights from Branch A.
    # ======================================================
    shared_stem = tf.keras.layers.Conv1D(
        filters=ENCODER_FILTERS_1,
        kernel_size=1,
        padding="same",
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(L2_REGULARIZATION),
        name="shared_feature_interaction_conv",
    )(normalized)

    shared_stage_1 = temporal_residual_block(
        inputs=shared_stem,
        filters=ENCODER_FILTERS_1,
        block_name="shared_temporal_block_1",
    )

    shared_stage_2 = temporal_residual_block(
        inputs=shared_stage_1,
        filters=ENCODER_FILTERS_2,
        block_name="shared_temporal_block_2",
    )

    shared_latent = temporal_residual_block(
        inputs=shared_stage_2,
        filters=LATENT_FILTERS,
        block_name="shared_latent_temporal_block",
    )

    shared_average = tf.keras.layers.GlobalAveragePooling1D(
        name="shared_temporal_average_pool",
    )(shared_latent)

    shared_maximum = tf.keras.layers.GlobalMaxPooling1D(
        name="shared_temporal_max_pool",
    )(shared_latent)

    temporal_summary = tf.keras.layers.Concatenate(
        name="temporal_summary",
    )([shared_average, shared_maximum])

    fused_features = tf.keras.layers.Concatenate(
        name="context_temporal_fusion",
    )([temporal_summary, context])

    fused_features = tf.keras.layers.Dense(
        24,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(L2_REGULARIZATION),
        name="shared_fusion_dense",
    )(fused_features)

    fused_features = tf.keras.layers.Dropout(
        DROPOUT_RATE, name="shared_fusion_dropout",
    )(fused_features)

    anomaly_features = tf.keras.layers.Dense(
        ANOMALY_DENSE_UNITS, activation="relu", name="anomaly_dense",
    )(fused_features)

    anomaly_output = tf.keras.layers.Dense(
        1, activation="sigmoid", name="anomaly_output",
    )(anomaly_features)

    class_features = tf.keras.layers.Dense(
        CLASS_DENSE_UNITS, activation="relu", name="class_dense",
    )(fused_features)

    class_output = tf.keras.layers.Dense(
        N_CLASSES, activation="softmax", name="class_output",
    )(class_features)

    model = tf.keras.Model(
        inputs=input_layer,
        outputs={
            "reconstruction_output": reconstruction_output,
            "anomaly_output": anomaly_output,
            "class_output": class_output,
        },
        name="dual_encoder_context_aware_temporal_cnn_safe",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,
            clipnorm=1.0,
        ),
        loss={
            "reconstruction_output": window_mse_loss,
            "anomaly_output": tf.keras.losses.BinaryCrossentropy(),
            "class_output": tf.keras.losses.SparseCategoricalCrossentropy(),
        },
        loss_weights={
            "reconstruction_output": RECON_LOSS_WEIGHT,
            "anomaly_output": ANOMALY_LOSS_WEIGHT,
            "class_output": CLASS_LOSS_WEIGHT,
        },
        metrics={
            "anomaly_output": [
                tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                tf.keras.metrics.AUC(name="auc"),
            ],
            "class_output": [
                tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            ],
        },
    )

    return model


preview_model = build_hybrid_model(
    normalization_mean=np.zeros(N_FEATURES, dtype=np.float32),
    normalization_variance=np.ones(N_FEATURES, dtype=np.float32),
)

preview_model.summary()

preview_parameters = preview_model.count_params()
preview_float32_kb = preview_parameters * 4 / 1024
preview_int8_weight_kb = preview_parameters / 1024

print("\nMODEL SIZE ESTIMATE")
print("Input:", WINDOW_LEN, "x", N_FEATURES)
print("Total parameters:", preview_parameters)
print("Estimated Float32 weights KB:", round(preview_float32_kb, 2))
print("Estimated INT8 weights KB:", round(preview_int8_weight_kb, 2))

del preview_model
tf.keras.backend.clear_session()

In [ ]:
# ==========================================================
# CELL 9 — training weights and dataset split helpers
# ==========================================================
def calculate_class_sample_weights(y_class):
    present_classes = np.unique(y_class)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_class,
    )

    class_weight_lookup = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            present_classes,
            class_weights,
        )
    }

    return np.asarray([
        class_weight_lookup[int(class_id)]
        for class_id in y_class
    ], dtype=np.float32)


def calculate_anomaly_sample_weights(y_anomaly):
    y_integer = y_anomaly.astype(np.int64)
    present_classes = np.unique(y_integer)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_integer,
    )

    lookup = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            present_classes,
            weights,
        )
    }

    return np.asarray([
        lookup[int(value)]
        for value in y_integer
    ], dtype=np.float32)


def recon_reference_for_groups(groups):
    """CHANGED: 1.0 for windows from normal-with-load files, else 0.0."""
    return np.asarray(
        [RECON_REFERENCE_BY_SESSION[int(g)] for g in groups],
        dtype=np.float32,
    )


def make_training_sample_weights(
    y_class,
    y_anomaly,
    groups,
):
    # CHANGED: AE trains ONLY on normal-with-load windows
    reconstruction_weights = recon_reference_for_groups(groups)

    class_weights = calculate_class_sample_weights(
        y_class
    )

    anomaly_weights = calculate_anomaly_sample_weights(
        y_anomaly
    )

    return {
        "reconstruction_output":
            reconstruction_weights,
        "anomaly_output":
            anomaly_weights,
        "class_output":
            class_weights,
    }


def split_train_validation_by_group(
    X,
    y_class,
    y_anomaly,
    files,
    starts,
    groups,
):
    unique_groups, first_indices = np.unique(
        groups,
        return_index=True,
    )
    group_labels = y_class[first_indices]

    class_group_counts = pd.Series(
        group_labels
    ).value_counts()

    if class_group_counts.min() < 2:
        raise RuntimeError(
            "Each class needs at least 2 sessions inside the training "
            "fold to create a stratified session-wise validation split. "
            f"Current session counts: {class_group_counts.to_dict()}"
        )

    validation_fraction = max(
        0.15,
        N_CLASSES / len(unique_groups),
    )
    validation_fraction = min(
        validation_fraction,
        0.40,
    )

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=validation_fraction,
        random_state=SEED,
    )

    group_train_pos, group_validation_pos = next(
        splitter.split(
            unique_groups,
            group_labels,
        )
    )

    train_group_ids = unique_groups[group_train_pos]
    validation_group_ids = unique_groups[group_validation_pos]

    train_index = np.flatnonzero(
        np.isin(groups, train_group_ids)
    )
    validation_index = np.flatnonzero(
        np.isin(groups, validation_group_ids)
    )

    if np.intersect1d(
        groups[train_index],
        groups[validation_index],
    ).size:
        raise RuntimeError(
            "Session leakage detected between training and validation."
        )

    train_data = (
        X[train_index],
        y_class[train_index],
        y_anomaly[train_index],
        files[train_index],
        starts[train_index],
        groups[train_index],
    )

    validation_data = (
        X[validation_index],
        y_class[validation_index],
        y_anomaly[validation_index],
        files[validation_index],
        starts[validation_index],
        groups[validation_index],
    )

    return train_data, validation_data

In [ ]:
# ==========================================================
# CELL 10 — fit one fold/model
# ==========================================================
def fit_model_with_group_validation(
    X_train_full,
    y_class_full,
    y_anomaly_full,
    files_full,
    starts_full,
    groups_full,
):
    (
        train_data,
        validation_data,
    ) = split_train_validation_by_group(
        X_train_full,
        y_class_full,
        y_anomaly_full,
        files_full,
        starts_full,
        groups_full,
    )

    (
        X_train,
        y_class_train,
        y_anomaly_train,
        _,
        _,
        groups_train,
    ) = train_data

    (
        X_validation,
        y_class_validation,
        y_anomaly_validation,
        _,
        _,
        groups_validation,
    ) = validation_data

    normalization_mean, normalization_variance = (
        calculate_normalization_statistics(
            X_train
        )
    )

    X_train_normalized = normalize_windows_numpy(
        X_train,
        normalization_mean,
        normalization_variance,
    )

    X_validation_normalized = normalize_windows_numpy(
        X_validation,
        normalization_mean,
        normalization_variance,
    )

    model = build_hybrid_model(
        normalization_mean,
        normalization_variance,
    )

    sample_weights = make_training_sample_weights(
        y_class_train,
        y_anomaly_train,
        groups_train,
    )

    validation_sample_weights = {
        "reconstruction_output":
            recon_reference_for_groups(groups_validation),  # CHANGED
        "anomaly_output":
            np.ones(
                len(y_anomaly_validation),
                dtype=np.float32,
            ),
        "class_output":
            np.ones(
                len(y_class_validation),
                dtype=np.float32,
            ),
    }

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_class_output_accuracy",
            mode="max",
            patience=20,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_class_output_loss",
            mode="min",
            factor=0.5,
            patience=7,
            min_lr=1e-5,
            verbose=1,
        ),
    ]

    history = model.fit(
        X_train,
        {
            "reconstruction_output":
                X_train_normalized,
            "anomaly_output":
                y_anomaly_train,
            "class_output":
                y_class_train,
        },
        sample_weight=sample_weights,
        validation_data=(
            X_validation,
            {
                "reconstruction_output":
                    X_validation_normalized,
                "anomaly_output":
                    y_anomaly_validation,
                "class_output":
                    y_class_validation,
            },
            validation_sample_weights,
        ),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )

    return (
        model,
        normalization_mean,
        normalization_variance,
        history,
    )

In [ ]:
# ==========================================================
# CELL 11 — decision fusion and conditional thresholds
# ==========================================================
def get_pwm_bin_index(mean_pwm):
    bin_index = np.searchsorted(
        PWM_BIN_EDGES,
        mean_pwm,
        side="right",
    ) - 1

    return int(
        np.clip(
            bin_index,
            0,
            len(PWM_BIN_NAMES) - 1,
        )
    )
# (fuse_decisions retained for CV-time compatibility; the
#  deployment-time fusion used for calibration/testing is
#  fuse_backend_decisions defined in CELL 22.)
def fuse_decisions(
    class_probabilities,
    anomaly_probabilities,
    reconstruction_mse,
    reconstruction_thresholds,
):
    class_ids = np.argmax(
        class_probabilities,
        axis=1,
    ).astype(np.int64)

    class_confidence = np.max(
        class_probabilities,
        axis=1,
    )

    reconstruction_anomaly_flags = (
        reconstruction_mse
        > reconstruction_thresholds
    )

    binary_anomaly_flags = (
        anomaly_probabilities
        >= ANOMALY_PROB_THRESHOLD
    )

    confident_known_fault_flags = (
        np.isin(
            class_ids,
            FAULT_CLASS_IDS,
        )
        & (
            class_confidence
            >= CLASS_CONF_THRESHOLD
        )
    )

    anomaly_flags = (
        binary_anomaly_flags
        | reconstruction_anomaly_flags
        | confident_known_fault_flags
    )

    final_labels = []

    for (
        class_id,
        confidence,
        binary_flag,
        reconstruction_flag,
        known_fault_flag,
    ) in zip(
        class_ids,
        class_confidence,
        binary_anomaly_flags,
        reconstruction_anomaly_flags,
        confident_known_fault_flags,
    ):
        predicted_label = FINAL_LABELS[
            int(class_id)
        ]

        if known_fault_flag:
            final_labels.append(predicted_label)
        elif binary_flag or reconstruction_flag:
            final_labels.append("unknown_fault")
        elif (
            class_id in SAFE_CLASS_IDS
            and confidence >= CLASS_CONF_THRESHOLD
        ):
            final_labels.append(predicted_label)
        else:
            final_labels.append("uncertain_safe")

    return (
        class_ids,
        class_confidence,
        binary_anomaly_flags,
        reconstruction_anomaly_flags,
        anomaly_flags,
        np.asarray(final_labels, dtype=object),
    )

In [ ]:
# ==========================================================
# CELL 12 — (CHANGED) no locked calibration/test split.
# Per decision: 5-fold CV (CELL 13/14) is the validation signal;
# the final shipped model trains on 100% of sessions. Threshold
# calibration in CELL 21 now uses the out-of-fold CV predictions
# instead of a held-out calibration set.
# ==========================================================
deployment_train_sessions = sessions  # 100% of sessions


def print_session_distribution(name, selected_sessions):
    counts = pd.Series([
        session["label"] for session in selected_sessions
    ]).value_counts().reindex(FINAL_LABELS, fill_value=0)

    print(f"\n{name}")
    print(counts)
    print("Total sessions:", len(selected_sessions))


print_session_distribution(
    "ALL SESSIONS (used for both CV and final 100% training)",
    deployment_train_sessions,
)

In [ ]:
# ==========================================================
# CELL 13 — five-fold file-wise cross-validation
# ==========================================================
cv_table = pd.DataFrame({
    "label": [
        session["class_id"]
        for session in deployment_train_sessions
    ],
    "group": [
        session["session_id"]
        for session in deployment_train_sessions
    ],
})

deployment_class_counts = pd.Series(
    cv_table["label"]
).value_counts()

EFFECTIVE_N_SPLITS = int(
    min(
        N_SPLITS,
        deployment_class_counts.min(),
    )
)

if EFFECTIVE_N_SPLITS < 2:
    raise RuntimeError(
        "At least two deployment-training sessions per class "
        "are required for cross-validation."
    )

print("Cross-validation folds:", EFFECTIVE_N_SPLITS)

splitter = StratifiedGroupKFold(
    n_splits=EFFECTIVE_N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
all_true = []
all_pred = []
all_anomaly_true = []
all_anomaly_pred = []
all_reconstruction_mse = []
all_mean_pwm = []
all_mean_irms = []
all_window_predictions = []
cv_best_epochs = []

for fold, (
    train_index,
    test_index,
) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    print("\n" + "=" * 70)
    print("FOLD", fold)
    print("=" * 70)

    train_sessions = [
        deployment_train_sessions[index]
        for index in train_index
    ]

    test_sessions = [
        deployment_train_sessions[index]
        for index in test_index
    ]

    (
        X_train_full,
        y_class_train_full,
        y_anomaly_train_full,
        train_files,
        train_starts,
        train_groups,
    ) = build_dataset_from_sessions(train_sessions)

    (
        X_test,
        y_class_test,
        y_anomaly_test,
        test_files,
        test_starts,
        _,
    ) = build_dataset_from_sessions(test_sessions)

    (
        model,
        normalization_mean,
        normalization_variance,
        history,
    ) = fit_model_with_group_validation(
        X_train_full,
        y_class_train_full,
        y_anomaly_train_full,
        train_files,
        train_starts,
        train_groups,
    )

    validation_accuracy_history = history.history.get(
        "val_class_output_accuracy", [],
    )
    if validation_accuracy_history:
        best_epoch = int(np.argmax(validation_accuracy_history) + 1)
    else:
        best_epoch = len(history.history.get("loss", []))

    cv_best_epochs.append(best_epoch)
    print("Best validation epoch:", best_epoch)

    X_test_normalized = normalize_windows_numpy(
        X_test, normalization_mean, normalization_variance,
    )

    prediction_dict = model.predict(
        X_test, batch_size=BATCH_SIZE, verbose=0,
    )

    reconstructed = prediction_dict["reconstruction_output"]
    anomaly_probabilities = prediction_dict["anomaly_output"].reshape(-1)
    class_probabilities = prediction_dict["class_output"]

    class_predictions = np.argmax(
        class_probabilities, axis=1,
    ).astype(np.int64)

    anomaly_predictions = (
        anomaly_probabilities >= ANOMALY_PROB_THRESHOLD
    ).astype(np.int64)

    reconstruction_mse = np.mean(
        np.square(reconstructed - X_test_normalized), axis=(1, 2),
    )

    pwm_feature_index = FEATURE_COLUMNS.index("PWM")
    irms_feature_index = FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)

    test_window_mean_pwm = np.mean(
        X_test[:, :, pwm_feature_index], axis=1,
    ).astype(np.float32)

    test_window_mean_irms = np.mean(
        X_test[:, :, irms_feature_index], axis=1,
    ).astype(np.float32)

    all_true.extend(y_class_test.tolist())
    all_pred.extend(class_predictions.tolist())
    all_anomaly_true.extend(y_anomaly_test.astype(np.int64).tolist())
    all_anomaly_pred.extend(anomaly_predictions.tolist())
    all_reconstruction_mse.extend(reconstruction_mse.tolist())
    all_mean_pwm.extend(test_window_mean_pwm.tolist())
    all_mean_irms.extend(test_window_mean_irms.tolist())

    fold_metrics = {
        "fold": fold,
        "accuracy": accuracy_score(y_class_test, class_predictions),
        "balanced_accuracy":
            balanced_accuracy_score(y_class_test, class_predictions),
        "macro_precision":
            precision_score(
                y_class_test, class_predictions,
                average="macro", zero_division=0,
            ),
        "macro_recall":
            recall_score(
                y_class_test, class_predictions,
                average="macro", zero_division=0,
            ),
        "macro_f1":
            f1_score(
                y_class_test, class_predictions,
                average="macro", zero_division=0,
            ),
        "anomaly_accuracy":
            accuracy_score(y_anomaly_test, anomaly_predictions),
        "anomaly_f1":
            f1_score(y_anomaly_test, anomaly_predictions, zero_division=0),
        "mean_reconstruction_mse": float(np.mean(reconstruction_mse)),
    }

    cv_metrics.append(fold_metrics)

    fold_predictions = pd.DataFrame({
        "fold": fold,
        "file": test_files,
        "start_row": test_starts,
        "mean_pwm": test_window_mean_pwm,
        "mean_irms": test_window_mean_irms,
        "true_class_id": y_class_test,
        "true_label": [FINAL_LABELS[i] for i in y_class_test],
        "pred_class_id": class_predictions,
        "pred_label": [FINAL_LABELS[i] for i in class_predictions],
        "class_confidence": np.max(class_probabilities, axis=1),
        "true_anomaly": y_anomaly_test.astype(np.int64),
        "anomaly_probability": anomaly_probabilities,
        "anomaly_prediction": anomaly_predictions,
        "reconstruction_mse": reconstruction_mse,
    })

    for class_id, label in enumerate(FINAL_LABELS):
        fold_predictions[f"probability_{label}"] = (
            class_probabilities[:, class_id]
        )

    all_window_predictions.append(fold_predictions)

    print(pd.Series(fold_metrics))

    print("\nFOLD CLASSIFICATION REPORT")
    print(
        classification_report(
            y_class_test, class_predictions,
            labels=np.arange(N_CLASSES),
            target_names=FINAL_LABELS, zero_division=0,
        )
    )

    fold_cm = confusion_matrix(
        y_class_test, class_predictions, labels=np.arange(N_CLASSES),
    )

    pd.DataFrame(
        fold_cm, index=FINAL_LABELS, columns=FINAL_LABELS,
    ).to_csv(
        os.path.join(CV_DIR, f"fold_{fold}_confusion_matrix.csv")
    )

    fig, axis = plt.subplots(figsize=(9, 8))
    ConfusionMatrixDisplay(
        fold_cm, display_labels=FINAL_LABELS,
    ).plot(
        ax=axis, values_format="d", xticks_rotation=45, colorbar=False,
    )
    axis.set_title(f"Fold {fold} — Session-wise Test Confusion Matrix")
    fig.tight_layout()
    fig.savefig(
        os.path.join(CV_DIR, f"fold_{fold}_confusion_matrix.png"), dpi=180,
    )
    plt.show()
    plt.close(fig)

    history_df = pd.DataFrame(history.history)
    history_df.to_csv(
        os.path.join(CV_DIR, f"fold_{fold}_training_history.csv"),
        index=False,
    )

    tf.keras.backend.clear_session()

In [ ]:
# ==========================================================
# CELL 14 — cross-validation report (model selection only)
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_predictions_df = pd.concat(all_window_predictions, ignore_index=True)

cv_metrics_df.to_csv(os.path.join(CV_DIR, "cv_metrics.csv"), index=False)
cv_predictions_df.to_csv(
    os.path.join(CV_DIR, "cv_window_predictions.csv"), index=False,
)

all_true = np.asarray(all_true, dtype=np.int64)
all_pred = np.asarray(all_pred, dtype=np.int64)
all_anomaly_true = np.asarray(all_anomaly_true, dtype=np.int64)
all_anomaly_pred = np.asarray(all_anomaly_pred, dtype=np.int64)

overall_cv_metrics = {
    "accuracy": accuracy_score(all_true, all_pred),
    "balanced_accuracy": balanced_accuracy_score(all_true, all_pred),
    "macro_precision":
        precision_score(all_true, all_pred, average="macro", zero_division=0),
    "macro_recall":
        recall_score(all_true, all_pred, average="macro", zero_division=0),
    "macro_f1":
        f1_score(all_true, all_pred, average="macro", zero_division=0),
    "anomaly_accuracy": accuracy_score(all_anomaly_true, all_anomaly_pred),
    "anomaly_f1":
        f1_score(all_anomaly_true, all_anomaly_pred, zero_division=0),
}

print("\nOVERALL DEVELOPMENT CROSS-VALIDATION RESULTS")
print(pd.Series(overall_cv_metrics))

print(
    classification_report(
        all_true, all_pred, labels=np.arange(N_CLASSES),
        target_names=FINAL_LABELS, zero_division=0,
    )
)

pd.DataFrame([overall_cv_metrics]).to_csv(
    os.path.join(CV_DIR, "overall_metrics.csv"), index=False,
)

overall_cv_cm = confusion_matrix(
    all_true, all_pred, labels=np.arange(N_CLASSES),
)

fig, axis = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(
    overall_cv_cm, display_labels=FINAL_LABELS,
).plot(ax=axis, values_format="d", xticks_rotation=45, colorbar=False)
axis.set_title("Development Session-wise Cross-Validation")
fig.tight_layout()
fig.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)
plt.show()
plt.close(fig)

In [ ]:
# ==========================================================
# CELL 15 — build the 100% training dataset
# (CHANGED: no more separate calibration/test session builds —
# threshold calibration now uses out-of-fold CV predictions
# from CELL 13/14, computed on data the CV models never trained on.)
# ==========================================================
(
    X_deployment_train,
    y_class_deployment_train,
    y_anomaly_deployment_train,
    deployment_train_files,
    deployment_train_starts,
    deployment_train_groups,
) = build_dataset_from_sessions(sessions)  # 100% of data

deployment_mean, deployment_variance = (
    calculate_normalization_statistics(X_deployment_train)
)

X_deployment_train_normalized = normalize_windows_numpy(
    X_deployment_train, deployment_mean, deployment_variance,
)

FINAL_EPOCHS = int(np.median(cv_best_epochs))
FINAL_EPOCHS = max(FINAL_EPOCHS, 1)

print("\nFINAL DEPLOYMENT DATA (100% of sessions)")
print("Training windows:", len(X_deployment_train))
print("Final epochs from median CV best epoch:", FINAL_EPOCHS)

In [ ]:
# ==========================================================
# CELL 16 — train final deployment Keras Float32 model
# ==========================================================
final_model = build_hybrid_model(deployment_mean, deployment_variance)

deployment_sample_weights = make_training_sample_weights(
    y_class_deployment_train, y_anomaly_deployment_train,
    deployment_train_groups,
)

final_history = final_model.fit(
    X_deployment_train,
    {
        "reconstruction_output": X_deployment_train_normalized,
        "anomaly_output": y_anomaly_deployment_train,
        "class_output": y_class_deployment_train,
    },
    sample_weight=deployment_sample_weights,
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
    shuffle=True,
)

pd.DataFrame(final_history.history).to_csv(
    os.path.join(FINAL_DIR, "final_training_history.csv"), index=False,
)

In [ ]:
# ==========================================================
# CELL 17 — save Keras and ONNX models
# ==========================================================
KERAS_PATH = os.path.abspath(os.path.join(
    FINAL_DIR, "compact_conv1d_ae_anomaly_classifier_safe.keras",
))
H5_PATH = os.path.abspath(os.path.join(
    FINAL_DIR, "compact_conv1d_ae_anomaly_classifier_safe.h5",
))
ONNX_PATH = os.path.abspath(os.path.join(
    FINAL_DIR, "compact_conv1d_ae_anomaly_classifier_safe.onnx",
))
LABELS_PATH = os.path.abspath(os.path.join(FINAL_DIR, "labels.txt"))

final_model.save(KERAS_PATH)
final_model.save(H5_PATH)

with open(LABELS_PATH, "w", encoding="utf-8") as file:
    file.write("\n".join(FINAL_LABELS))

print("Saved:", KERAS_PATH)
print("Saved:", H5_PATH)
print("Saved:", LABELS_PATH)

try:
    import tf2onnx

    input_signature = (
        tf.TensorSpec(
            shape=(None, WINDOW_LEN, N_FEATURES),
            dtype=tf.float32,
            name="motor_window",
        ),
    )

    tf2onnx.convert.from_keras(
        final_model,
        input_signature=input_signature,
        opset=13,
        output_path=ONNX_PATH,
    )

    print("Saved:", ONNX_PATH)
except Exception as error:
    print("WARNING: ONNX export failed:", error)

In [ ]:
# ==========================================================
# CELL 18 — export TFLite Float32 and TFLite INT8
# ==========================================================
TFLITE_FLOAT32_PATH = os.path.abspath(os.path.join(
    FINAL_DIR,
    "compact_conv1d_ae_anomaly_classifier_safe_float32.tflite",
))

TFLITE_INT8_PATH = os.path.abspath(os.path.join(
    FINAL_DIR,
    "compact_conv1d_ae_anomaly_classifier_safe_int8.tflite",
))

float32_converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
tflite_float32_model = float32_converter.convert()

with open(TFLITE_FLOAT32_PATH, "wb") as file:
    file.write(tflite_float32_model)


def representative_dataset():
    sample_count = min(1000, len(X_deployment_train))

    rng = np.random.default_rng(SEED)
    sample_indices = rng.choice(
        len(X_deployment_train), size=sample_count, replace=False,
    )

    for index in sample_indices:
        yield [X_deployment_train[index:index + 1].astype(np.float32)]


int8_converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter.representative_dataset = representative_dataset
int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]
int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

tflite_int8_model = int8_converter.convert()

with open(TFLITE_INT8_PATH, "wb") as file:
    file.write(tflite_int8_model)

print("Saved Float32 TFLite:", TFLITE_FLOAT32_PATH)
print("Saved INT8 TFLite:", TFLITE_INT8_PATH)
print(
    "Float32 TFLite KB:",
    round(os.path.getsize(TFLITE_FLOAT32_PATH) / 1024, 2),
)
print(
    "INT8 TFLite KB:",
    round(os.path.getsize(TFLITE_INT8_PATH) / 1024, 2),
)

# ==========================================================
# VERIFY: confirm no SpaceToBatchND/BatchToSpaceND remain
# ==========================================================
interpreter_check = tf.lite.Interpreter(model_path=TFLITE_FLOAT32_PATH)
interpreter_check.allocate_tensors()
op_details = interpreter_check._get_ops_details()
op_names = sorted({op["op_name"] for op in op_details})
print("\nOPS IN EXPORTED GRAPH:")
for name in op_names:
    print(" ", name)
forbidden_ops = {"SPACE_TO_BATCH_ND", "BATCH_TO_SPACE_ND"}
found_forbidden = forbidden_ops.intersection(op_names)
if found_forbidden:
    print(
        "\nWARNING: forbidden dilation-lowering ops still present:",
        found_forbidden,
    )
else:
    print(
        "\nOK: no SpaceToBatchND/BatchToSpaceND ops in the exported graph."
    )

In [ ]:
# ==========================================================
# CELL 19 — backend inference utilities
# ==========================================================
def ensure_prediction_dict(prediction):
    if isinstance(prediction, dict):
        return {key: np.asarray(value) for key, value in prediction.items()}

    if isinstance(prediction, (list, tuple)):
        result = {}

        for value in prediction:
            value = np.asarray(value)

            if value.shape[-2:] == (WINDOW_LEN, N_FEATURES):
                result["reconstruction_output"] = value
            elif value.shape[-1] == N_CLASSES:
                result["class_output"] = value
            elif value.shape[-1] == 1:
                result["anomaly_output"] = value

        return result

    raise TypeError("Unsupported prediction output type.")


def predict_keras_float32(model, X):
    prediction = model.predict(X, batch_size=BATCH_SIZE, verbose=0)
    prediction = ensure_prediction_dict(prediction)

    required = {
        "reconstruction_output", "anomaly_output", "class_output",
    }

    if set(prediction) != required:
        raise RuntimeError(
            f"Keras output mapping failed. Found: {list(prediction)}"
        )

    return {
        "reconstruction_output":
            prediction["reconstruction_output"].astype(np.float32),
        "anomaly_output":
            prediction["anomaly_output"].reshape(-1).astype(np.float32),
        "class_output":
            prediction["class_output"].astype(np.float32),
    }


def quantize_tensor(values, detail):
    dtype = detail["dtype"]

    if dtype == np.float32:
        return values.astype(np.float32)

    scale, zero_point = detail["quantization"]

    if scale <= 0:
        raise RuntimeError(
            f"Invalid quantization scale for {detail['name']}: {scale}"
        )

    info = np.iinfo(dtype)
    quantized = np.round(values / scale + zero_point)
    quantized = np.clip(quantized, info.min, info.max)

    return quantized.astype(dtype)


def dequantize_tensor(values, detail):
    if detail["dtype"] == np.float32:
        return values.astype(np.float32)

    scale, zero_point = detail["quantization"]

    if scale <= 0:
        raise RuntimeError(
            f"Invalid output quantization scale for "
            f"{detail['name']}: {scale}"
        )

    return (
        (values.astype(np.float32) - float(zero_point)) * float(scale)
    ).astype(np.float32)


def identify_output_role(detail):
    shape = tuple(int(v) for v in detail["shape"])
    element_count = int(np.prod(shape[1:]))

    if element_count == WINDOW_LEN * N_FEATURES:
        return "reconstruction_output"
    if element_count == N_CLASSES:
        return "class_output"
    if element_count == 1:
        return "anomaly_output"

    raise RuntimeError(
        f"Cannot identify TFLite output from "
        f"name={detail['name']}, shape={shape}"
    )


def predict_tflite(model_path, X):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    if len(input_details) != 1:
        raise RuntimeError("Expected exactly one TFLite input.")

    input_detail = input_details[0]

    outputs = {
        "reconstruction_output": [],
        "anomaly_output": [],
        "class_output": [],
    }

    for start in range(0, len(X), BATCH_SIZE):
        batch = X[start:start + BATCH_SIZE].astype(np.float32)

        required_shape = list(input_detail["shape"])
        required_shape[0] = len(batch)

        if list(input_detail["shape"]) != required_shape:
            interpreter.resize_tensor_input(
                input_detail["index"], required_shape, strict=False,
            )
            interpreter.allocate_tensors()
            input_detail = interpreter.get_input_details()[0]
            output_details = interpreter.get_output_details()

        interpreter.set_tensor(
            input_detail["index"], quantize_tensor(batch, input_detail),
        )

        interpreter.invoke()

        for detail in output_details:
            role = identify_output_role(detail)
            raw_output = interpreter.get_tensor(detail["index"])
            output = dequantize_tensor(raw_output, detail)

            if role == "reconstruction_output":
                output = output.reshape(-1, WINDOW_LEN, N_FEATURES)
            elif role == "class_output":
                output = output.reshape(-1, N_CLASSES)
            else:
                output = output.reshape(-1)

            outputs[role].append(output)

    return {
        role: np.concatenate(values, axis=0).astype(np.float32)
        for role, values in outputs.items()
    }


def print_tflite_interface(model_path):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    print("\nMODEL:", model_path)

    print("INPUTS")
    for detail in interpreter.get_input_details():
        print(
            detail["name"], detail["shape"],
            detail["dtype"], detail["quantization"],
        )

    print("OUTPUTS")
    for detail in interpreter.get_output_details():
        print(
            identify_output_role(detail), detail["name"],
            detail["shape"], detail["dtype"], detail["quantization"],
        )


print_tflite_interface(TFLITE_FLOAT32_PATH)
print_tflite_interface(TFLITE_INT8_PATH)

In [ ]:
# ==========================================================
# CELL 20 — reconstruction and operating-context utilities
# (CHANGED: build_threshold_bundle now also records a per-bin
# "local baseline" (median reconstruction MSE), which severity
# scoring needs. ADDED: severity_level_and_action() and
# get_local_normal_baseline(), matching the DANN notebook.
# CHANGED: calibration_session_count now derives from cv_table
# since there is no more separate locked calibration session set.)
# ==========================================================
PWM_FEATURE_INDEX = FEATURE_COLUMNS.index("PWM")
IRMS_FEATURE_INDEX = FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)


def calculate_backend_values(X, predictions):
    normalized_target = normalize_windows_numpy(
        X, deployment_mean, deployment_variance,
    )

    reconstruction_mse = np.mean(
        np.square(
            predictions["reconstruction_output"] - normalized_target
        ),
        axis=(1, 2),
    ).astype(np.float32)

    mean_pwm = np.mean(
        X[:, :, PWM_FEATURE_INDEX], axis=1,
    ).astype(np.float32)

    mean_irms = np.mean(
        X[:, :, IRMS_FEATURE_INDEX], axis=1,
    ).astype(np.float32)

    return reconstruction_mse, mean_pwm, mean_irms


def calibrate_anomaly_probability_threshold(y_true, probabilities):
    y_true = np.asarray(y_true, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float32)

    candidates = np.unique(np.concatenate([
        np.linspace(0.05, 0.95, 181, dtype=np.float32),
        probabilities,
    ]))

    best_threshold = 0.50
    best_score = -1.0
    best_distance = np.inf

    for threshold in candidates:
        predictions = (probabilities >= threshold).astype(np.int64)
        score = balanced_accuracy_score(y_true, predictions)
        distance = abs(float(threshold) - 0.50)

        if (
            score > best_score
            or (np.isclose(score, best_score) and distance < best_distance)
        ):
            best_score = float(score)
            best_threshold = float(threshold)
            best_distance = distance

    return best_threshold, best_score


def severity_level_and_action(severity_ratio):
    """
    ADDED — matches the DANN notebook's severity tiering exactly,
    so both models produce the same output granularity for a fair
    comparison.
    """
    if severity_ratio < SEVERITY_MILD_MULTIPLIER:
        return "normal", "No action needed."
    elif severity_ratio < SEVERITY_MODERATE_MULTIPLIER:
        return "mild", "Log and monitor; no immediate action."
    elif severity_ratio < SEVERITY_SEVERE_MULTIPLIER:
        return "moderate", "Schedule inspection soon."
    else:
        return "severe", "Stop motor / inspect immediately."


def build_threshold_bundle(
    backend_name, y_class, y_anomaly, reconstruction_mse,
    mean_pwm, mean_irms, anomaly_probabilities,
):
    y_class = np.asarray(y_class, dtype=np.int64)
    y_anomaly = np.asarray(y_anomaly, dtype=np.int64)
    reconstruction_mse = np.asarray(reconstruction_mse, dtype=np.float32)
    mean_pwm = np.asarray(mean_pwm, dtype=np.float32)
    mean_irms = np.asarray(mean_irms, dtype=np.float32)

    safe_mask = np.isin(y_class, SAFE_CLASS_IDS)

    if not np.any(safe_mask):
        raise RuntimeError("Calibration set contains no safe windows.")

    global_safe_threshold = float(
        np.quantile(reconstruction_mse[safe_mask], RECON_QUANTILE)
    )

    # ADDED: global baseline for severity scoring fallback.
    global_normal_baseline = float(
        np.median(reconstruction_mse[safe_mask])
    )

    state_thresholds = {}

    for state_label in ["motor_off", "turning_off", "turning_on"]:
        state_errors = reconstruction_mse[
            y_class == LABEL_TO_ID[state_label]
        ]

        if len(state_errors) >= MIN_THRESHOLD_WINDOWS:
            threshold = float(np.quantile(state_errors, RECON_QUANTILE))
        else:
            threshold = global_safe_threshold

        state_thresholds[state_label] = threshold

    normal_mask = y_class == LABEL_TO_ID["normal"]

    if not np.any(normal_mask):
        raise RuntimeError("Calibration set contains no normal windows.")

    normal_errors = reconstruction_mse[normal_mask]
    normal_pwm = mean_pwm[normal_mask]
    normal_irms = mean_irms[normal_mask]

    global_normal_threshold = float(
        np.quantile(normal_errors, RECON_QUANTILE)
    )
    global_normal_load_split = float(
        np.quantile(normal_irms, LOAD_SPLIT_QUANTILE)
    )

    pwm_load_splits = {}
    pwm_thresholds = {}
    pwm_load_thresholds = {}
    # ADDED: per-bin local baselines (median), needed for severity ratios.
    pwm_local_baselines = {}
    threshold_rows = []

    for bin_index, bin_name in enumerate(PWM_BIN_NAMES):
        lower_pwm = float(PWM_BIN_EDGES[bin_index])
        upper_pwm = float(PWM_BIN_EDGES[bin_index + 1])

        pwm_mask = (normal_pwm >= lower_pwm) & (normal_pwm < upper_pwm)
        pwm_errors = normal_errors[pwm_mask]
        pwm_irms = normal_irms[pwm_mask]

        if len(pwm_errors) >= MIN_THRESHOLD_WINDOWS:
            pwm_threshold = float(np.quantile(pwm_errors, RECON_QUANTILE))
            pwm_baseline = float(np.median(pwm_errors))
            load_split = float(np.quantile(pwm_irms, LOAD_SPLIT_QUANTILE))
        else:
            pwm_threshold = global_normal_threshold
            pwm_baseline = global_normal_baseline
            load_split = global_normal_load_split

        pwm_load_splits[bin_name] = load_split
        pwm_thresholds[bin_name] = pwm_threshold
        pwm_local_baselines[bin_name] = pwm_baseline
        pwm_load_thresholds[bin_name] = {}

        for load_state in LOAD_STATE_NAMES:
            if load_state == "low_load":
                load_mask = pwm_mask & (normal_irms < load_split)
            else:
                load_mask = pwm_mask & (normal_irms >= load_split)

            load_errors = normal_errors[load_mask]

            if len(load_errors) >= MIN_THRESHOLD_WINDOWS:
                load_threshold = float(
                    np.quantile(load_errors, RECON_QUANTILE)
                )
            else:
                load_threshold = pwm_threshold

            pwm_load_thresholds[bin_name][load_state] = load_threshold

            threshold_rows.append({
                "backend": backend_name,
                "pwm_bin": bin_name,
                "load_state": load_state,
                "window_count": int(len(load_errors)),
                "irms_split": float(load_split),
                "reconstruction_threshold": float(load_threshold),
                "local_baseline": float(pwm_baseline),
            })

    # Use the explicitly selected deployment threshold everywhere:
    # notebook decisions, final evaluation, and STM32 header.
    anomaly_threshold = float(ANOMALY_PROB_THRESHOLD)

    binary_predictions = (
        anomaly_probabilities >= anomaly_threshold
    ).astype(np.int64)

    calibration_balanced_accuracy = float(
        balanced_accuracy_score(y_anomaly, binary_predictions)
    )

    bundle = {
        "backend": backend_name,
        "reconstruction_quantile": RECON_QUANTILE,
        "global_reconstruction_fallback": global_safe_threshold,
        "global_normal_reconstruction_threshold": global_normal_threshold,
        "global_normal_baseline": global_normal_baseline,
        "state_reconstruction_thresholds": state_thresholds,
        "pwm_bin_edges": PWM_BIN_EDGES.tolist(),
        "pwm_bin_names": PWM_BIN_NAMES,
        "load_proxy_feature": LOAD_PROXY_FEATURE,
        "global_normal_load_split": global_normal_load_split,
        "pwm_load_splits": pwm_load_splits,
        "normal_pwm_reconstruction_thresholds": pwm_thresholds,
        "normal_pwm_local_baselines": pwm_local_baselines,
        "normal_pwm_load_reconstruction_thresholds": pwm_load_thresholds,
        "anomaly_probability_threshold": anomaly_threshold,
        "anomaly_calibration_balanced_accuracy":
            calibration_balanced_accuracy,
        "class_confidence_threshold": CLASS_CONF_THRESHOLD,
        "calibration_window_count": int(len(y_class)),
        "calibration_session_count": int(len(np.unique(cv_table["group"]))),
        "severity_mild_multiplier": SEVERITY_MILD_MULTIPLIER,
        "severity_moderate_multiplier": SEVERITY_MODERATE_MULTIPLIER,
        "severity_severe_multiplier": SEVERITY_SEVERE_MULTIPLIER,
    }

    return bundle, pd.DataFrame(threshold_rows)


def get_pwm_bin_index(mean_pwm):
    index = np.searchsorted(PWM_BIN_EDGES, mean_pwm, side="right") - 1
    return int(np.clip(index, 0, len(PWM_BIN_NAMES) - 1))


def get_conditional_threshold(bundle, predicted_class_id, mean_pwm, mean_irms):
    predicted_label = FINAL_LABELS[int(predicted_class_id)]
    global_fallback = float(bundle["global_reconstruction_fallback"])

    if predicted_label in ["motor_off", "turning_off", "turning_on"]:
        return float(
            bundle["state_reconstruction_thresholds"].get(
                predicted_label, global_fallback,
            )
        )

    bin_name = PWM_BIN_NAMES[get_pwm_bin_index(mean_pwm)]

    load_split = float(
        bundle["pwm_load_splits"].get(
            bin_name, bundle["global_normal_load_split"],
        )
    )

    load_state = "high_load" if mean_irms >= load_split else "low_load"

    return float(
        bundle["normal_pwm_load_reconstruction_thresholds"].get(
            bin_name, {},
        ).get(
            load_state,
            bundle["normal_pwm_reconstruction_thresholds"].get(
                bin_name, global_fallback,
            ),
        )
    )


def get_local_normal_baseline(bundle, mean_pwm):
    """ADDED — mirrors get_conditional_threshold, but returns the
    per-bin baseline used for severity ratio scoring."""
    bin_name = PWM_BIN_NAMES[get_pwm_bin_index(mean_pwm)]

    return float(
        bundle["normal_pwm_local_baselines"].get(
            bin_name, bundle["global_normal_baseline"],
        )
    )


def thresholds_for_windows(bundle, class_probabilities, mean_pwm, mean_irms):
    predicted_classes = np.argmax(class_probabilities, axis=1)

    return np.asarray([
        get_conditional_threshold(bundle, class_id, pwm, irms)
        for class_id, pwm, irms in zip(predicted_classes, mean_pwm, mean_irms)
    ], dtype=np.float32)


def local_baselines_for_windows(bundle, mean_pwm):
    """ADDED — vectorized helper for severity scoring."""
    return np.asarray([
        get_local_normal_baseline(bundle, pwm) for pwm in mean_pwm
    ], dtype=np.float32)

In [ ]:
# ==========================================================
# CELL 21 — calibrate thresholds from out-of-fold CV predictions
# (CHANGED: no locked calibration session split anymore. The CV
# predictions in cv_predictions_df are already genuinely held-out
# per fold, so they're a valid, unbiased source for threshold
# calibration — consistent with "5-fold CV is enough" validation.)
# ==========================================================
BACKEND_NAME = "tflite_float32"


def FLOAT32_PREDICTOR(X):
    return predict_tflite(TFLITE_FLOAT32_PATH, X)


print("\nCALIBRATING BACKEND:", BACKEND_NAME, "(from out-of-fold CV predictions)")

(
    FLOAT32_THRESHOLD_BUNDLE,
    FLOAT32_THRESHOLD_TABLE,
) = build_threshold_bundle(
    backend_name=BACKEND_NAME,
    y_class=cv_predictions_df["true_class_id"].to_numpy(),
    y_anomaly=cv_predictions_df["true_anomaly"].to_numpy(),
    reconstruction_mse=cv_predictions_df["reconstruction_mse"].to_numpy(),
    mean_pwm=cv_predictions_df["mean_pwm"].to_numpy(),
    mean_irms=cv_predictions_df["mean_irms"].to_numpy(),
    anomaly_probabilities=cv_predictions_df["anomaly_probability"].to_numpy(),
)

FLOAT32_THRESHOLD_JSON_PATH = os.path.join(
    FINAL_DIR, "thresholds_tflite_float32.json",
)

with open(FLOAT32_THRESHOLD_JSON_PATH, "w", encoding="utf-8") as file:
    json.dump(FLOAT32_THRESHOLD_BUNDLE, file, indent=2)

FLOAT32_THRESHOLD_TABLE.to_csv(
    os.path.join(FINAL_DIR, "conditional_thresholds_tflite_float32.csv"),
    index=False,
)

print(
    "Global reconstruction threshold:",
    FLOAT32_THRESHOLD_BUNDLE["global_reconstruction_fallback"],
)
print(
    "Anomaly probability threshold:",
    FLOAT32_THRESHOLD_BUNDLE["anomaly_probability_threshold"],
)

In [ ]:
# ==========================================================
# CELL 22 — decision fusion + sanity check on the 100% training set
# (CHANGED: there is no more held-out test set. The honest
# generalization estimate is the 5-fold CV report in CELL 14.
# This cell now only runs a sanity check on the model's own
# training data before export — useful to catch broken exports,
# not to be read as a generalization metric.)
# ==========================================================
def fuse_backend_decisions(
    class_probabilities, anomaly_probabilities,
    reconstruction_mse, reconstruction_thresholds, local_baselines,
    threshold_bundle,
):
    class_ids = np.argmax(class_probabilities, axis=1).astype(np.int64)
    class_confidence = np.max(
        class_probabilities, axis=1
    ).astype(np.float32)

    # Top-class probability minus second-highest probability
    sorted_probabilities = np.sort(class_probabilities, axis=1)
    class_margin = (
        sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
    ).astype(np.float32)

    classifier_reliable = (
        (class_confidence >= float(
            threshold_bundle["class_confidence_threshold"]
        ))
        | (class_margin >= 0.25)
    )

    # If the class is unreliable, use global reconstruction threshold.
    effective_reconstruction_thresholds = np.where(
        classifier_reliable,
        reconstruction_thresholds,
        float(threshold_bundle["global_reconstruction_fallback"]),
    ).astype(np.float32)

    binary_anomaly_flags = (
        anomaly_probabilities
        >= float(threshold_bundle["anomaly_probability_threshold"])
    )

    reconstruction_anomaly_flags = (
        reconstruction_mse > effective_reconstruction_thresholds
    )

    # MSE/APROB provide independent anomaly evidence.
    evidence_anomaly_flags = (
        binary_anomaly_flags | reconstruction_anomaly_flags
    )

    reliable_fault_class_flags = (
        np.isin(class_ids, FAULT_CLASS_IDS)
        & classifier_reliable
    )

    reliable_safe_class_flags = (
        np.isin(class_ids, SAFE_CLASS_IDS)
        & classifier_reliable
    )

    # A reliable fault class still raises an alarm,
    # but it is named only when MSE/APROB also confirms anomaly.
    final_anomaly_flags = (
        evidence_anomaly_flags | reliable_fault_class_flags
    )

    safe_baselines = np.where(local_baselines > 0, local_baselines, 1.0)
    severity_ratios = reconstruction_mse / safe_baselines

    severity_labels = []
    severity_actions = []

    for ratio in severity_ratios:
        label, action = severity_level_and_action(float(ratio))
        severity_labels.append(label)
        severity_actions.append(action)

    final_labels = []

    for (
        class_id,
        evidence_anomaly,
        reliable_fault,
        reliable_safe,
    ) in zip(
        class_ids,
        evidence_anomaly_flags,
        reliable_fault_class_flags,
        reliable_safe_class_flags,
    ):
        predicted_label = FINAL_LABELS[int(class_id)]

        if evidence_anomaly:
            # MSE or binary head confirms anomaly.
            if reliable_fault:
                final_labels.append(predicted_label)
            else:
                final_labels.append("unknown_fault")

        else:
            # Both MSE and aprob say normal.
            if reliable_safe:
                final_labels.append(predicted_label)

            elif reliable_fault:
                # Classifier suspects a fault, but anomaly heads
                # do not confirm which fault it is.
                final_labels.append("unknown_fault")

            else:
                final_labels.append("uncertain_safe")

    return {
        "class_ids": class_ids,
        "class_confidence": class_confidence,
        "class_margin": class_margin,
        "classifier_reliable": classifier_reliable,
        "binary_anomaly_flags": binary_anomaly_flags,
        "reconstruction_anomaly_flags": reconstruction_anomaly_flags,
        "effective_reconstruction_thresholds":
            effective_reconstruction_thresholds,
        "final_anomaly_flags": final_anomaly_flags,
        "final_labels": np.asarray(final_labels, dtype=object),
        "severity_ratio": severity_ratios.astype(np.float32),
        "severity_label": np.asarray(severity_labels, dtype=object),
        "recommended_action": np.asarray(
            severity_actions, dtype=object
        ),
    }


def evaluate_float32_backend(
    predictions, X, y_class, y_anomaly, files, starts, groups,
):
    (
        reconstruction_mse, mean_pwm, mean_irms,
    ) = calculate_backend_values(X, predictions)

    reconstruction_thresholds = thresholds_for_windows(
        FLOAT32_THRESHOLD_BUNDLE, predictions["class_output"],
        mean_pwm, mean_irms,
    )

    # ADDED: local baselines per window for severity scoring.
    local_baselines = local_baselines_for_windows(
        FLOAT32_THRESHOLD_BUNDLE, mean_pwm,
    )

    decisions = fuse_backend_decisions(
        class_probabilities=predictions["class_output"],
        anomaly_probabilities=predictions["anomaly_output"],
        reconstruction_mse=reconstruction_mse,
        reconstruction_thresholds=reconstruction_thresholds,
        local_baselines=local_baselines,
        threshold_bundle=FLOAT32_THRESHOLD_BUNDLE,
    )

    class_ids = decisions["class_ids"]

    binary_predictions = (
        predictions["anomaly_output"]
        >= float(FLOAT32_THRESHOLD_BUNDLE["anomaly_probability_threshold"])
    ).astype(np.int64)

    final_anomaly_predictions = (
        decisions["final_anomaly_flags"].astype(np.int64)
    )

    metrics = {
        "backend": BACKEND_NAME,
        "window_count": int(len(X)),
        "session_count": int(len(np.unique(groups))),
        "class_accuracy": accuracy_score(y_class, class_ids),
        "class_balanced_accuracy":
            balanced_accuracy_score(y_class, class_ids),
        "class_macro_f1":
            f1_score(y_class, class_ids, average="macro", zero_division=0),
        "binary_head_accuracy":
            accuracy_score(y_anomaly, binary_predictions),
        "binary_head_f1":
            f1_score(y_anomaly, binary_predictions, zero_division=0),
        "fused_anomaly_accuracy":
            accuracy_score(y_anomaly, final_anomaly_predictions),
        "fused_anomaly_f1":
            f1_score(y_anomaly, final_anomaly_predictions, zero_division=0),
        "safe_false_anomaly_rate": float(np.mean(
            final_anomaly_predictions[
                np.asarray(y_anomaly, dtype=np.int64) == 0
            ]
        )),
        "fault_detection_rate": float(np.mean(
            final_anomaly_predictions[
                np.asarray(y_anomaly, dtype=np.int64) == 1
            ]
        )),
        "mean_reconstruction_mse": float(np.mean(reconstruction_mse)),
        "severe_window_fraction": float(
            np.mean(decisions["severity_label"] == "severe")
        ),
    }

    result = pd.DataFrame({
        "backend": BACKEND_NAME,
        "file": files,
        "session_id": groups,
        "start_row": starts,
        "true_class_id": y_class,
        "true_label": [FINAL_LABELS[i] for i in y_class],
        "pred_class_id": class_ids,
        "pred_label": [FINAL_LABELS[i] for i in class_ids],
        "class_confidence": decisions["class_confidence"],
        "true_anomaly": np.asarray(y_anomaly, dtype=np.int64),
        "anomaly_probability": predictions["anomaly_output"],
        "mean_pwm": mean_pwm,
        "mean_irms": mean_irms,
        "reconstruction_mse": reconstruction_mse,
        "reconstruction_threshold": decisions["effective_reconstruction_thresholds"],
        "local_baseline": local_baselines,
        "severity_ratio": decisions["severity_ratio"],
        "severity_label": decisions["severity_label"],
        "recommended_action": decisions["recommended_action"],
        "final_anomaly_flag": final_anomaly_predictions,
        "final_decision": decisions["final_labels"],
    })

    for class_id, label in enumerate(FINAL_LABELS):
        result[f"probability_{label}"] = (
            predictions["class_output"][:, class_id]
        )

    return metrics, result


print("\nSANITY CHECK ONLY (training-set fit, NOT a generalization metric).")
print("See CELL 14 for the honest 5-fold cross-validation metrics.\n")

FLOAT32_TRAIN_PREDICTIONS = FLOAT32_PREDICTOR(X_deployment_train)

(
    FLOAT32_TRAIN_METRICS,
    FLOAT32_TRAIN_RESULTS,
) = evaluate_float32_backend(
    predictions=FLOAT32_TRAIN_PREDICTIONS,
    X=X_deployment_train,
    y_class=y_class_deployment_train,
    y_anomaly=y_anomaly_deployment_train,
    files=deployment_train_files,
    starts=deployment_train_starts,
    groups=deployment_train_groups,
)

FLOAT32_TRAIN_RESULTS.to_csv(
    os.path.join(FINAL_DIR, "train_predictions_tflite_float32.csv"),
    index=False,
)

pd.DataFrame([FLOAT32_TRAIN_METRICS]).to_csv(
    os.path.join(FINAL_DIR, "train_metrics_tflite_float32_SANITY_ONLY.csv"),
    index=False,
)

print(pd.Series(FLOAT32_TRAIN_METRICS))

In [ ]:
# ==========================================================
# CELL 23 — export Float32 and INT8 model files as C headers
# ==========================================================
def write_binary_model_header(binary_model_path, header_path, array_name):
    with open(binary_model_path, "rb") as file:
        model_bytes = file.read()

    include_guard = array_name.upper() + "_H"

    with open(header_path, "w", encoding="utf-8") as file:
        file.write(f"#ifndef {include_guard}\n")
        file.write(f"#define {include_guard}\n\n")
        file.write("#include <stdint.h>\n")
        file.write("#include <stddef.h>\n\n")
        file.write("#if defined(__GNUC__)\n")
        file.write(
            "#define MODEL_ALIGN __attribute__((aligned(16)))\n"
        )
        file.write("#else\n")
        file.write("#define MODEL_ALIGN\n")
        file.write("#endif\n\n")

        file.write(
            f"static const MODEL_ALIGN uint8_t {array_name}"
            f"[{len(model_bytes)}] = {{\n"
        )

        for start in range(0, len(model_bytes), 12):
            chunk = model_bytes[start:start + 12]
            file.write(
                "    " + ", ".join(f"0x{b:02X}" for b in chunk) + ",\n"
            )

        file.write("};\n\n")
        file.write(
            f"static const size_t {array_name}_len = {len(model_bytes)}u;\n\n"
        )
        file.write(f"#endif /* {include_guard} */\n")

    return len(model_bytes)


FLOAT32_MODEL_HEADER_PATH = os.path.join(
    FINAL_DIR, "model_tflite_float32_safe.h",
)
INT8_MODEL_HEADER_PATH = os.path.join(
    FINAL_DIR, "model_tflite_int8_safe.h",
)

write_binary_model_header(
    TFLITE_FLOAT32_PATH, FLOAT32_MODEL_HEADER_PATH,
    "motor_model_tflite_float32_safe",
)
write_binary_model_header(
    TFLITE_INT8_PATH, INT8_MODEL_HEADER_PATH,
    "motor_model_tflite_int8_safe",
)

print("Saved Float32 header:", FLOAT32_MODEL_HEADER_PATH)
print("Saved INT8 header:", INT8_MODEL_HEADER_PATH)

In [ ]:
# ==========================================================
# CELL 24 — export firmware parameter header (TFLite Float32)
# (CHANGED: write_params_header now also embeds severity
# multipliers and per-bin local baselines, matching the DANN
# notebook's firmware header.)
# ==========================================================
def c_float(value):
    text = f"{float(value):.9g}"
    if "." not in text and "e" not in text and "E" not in text:
        text += ".0"
    return f"{text}f"


def get_tflite_quantization_metadata(model_path):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]
    output_metadata = {}

    for detail in interpreter.get_output_details():
        role = identify_output_role(detail)
        scale, zero_point = detail["quantization"]

        output_metadata[role] = {
            "dtype": np.dtype(detail["dtype"]).name,
            "scale": float(scale),
            "zero_point": int(zero_point),
        }

    input_scale, input_zero_point = input_detail["quantization"]

    return {
        "input": {
            "dtype": np.dtype(input_detail["dtype"]).name,
            "scale": float(input_scale),
            "zero_point": int(input_zero_point),
        },
        "outputs": output_metadata,
    }


def write_params_header(
    header_path, include_guard, bundle, quantization_metadata=None,
):
    low_thresholds = [
        bundle["normal_pwm_load_reconstruction_thresholds"][bin_name]["low_load"]
        for bin_name in PWM_BIN_NAMES
    ]
    high_thresholds = [
        bundle["normal_pwm_load_reconstruction_thresholds"][bin_name]["high_load"]
        for bin_name in PWM_BIN_NAMES
    ]
    load_splits = [
        bundle["pwm_load_splits"][bin_name] for bin_name in PWM_BIN_NAMES
    ]
    # ADDED
    local_baselines = [
        bundle["normal_pwm_local_baselines"][bin_name] for bin_name in PWM_BIN_NAMES
    ]

    with open(header_path, "w", encoding="utf-8") as file:
        file.write(f"#ifndef {include_guard}\n")
        file.write(f"#define {include_guard}\n\n")

        file.write(f"#define MOTOR_MODEL_WINDOW_LEN {WINDOW_LEN}\n")
        file.write(f"#define MOTOR_MODEL_N_FEATURES {N_FEATURES}\n")
        file.write(
            f"#define MOTOR_MODEL_INPUT_SIZE {WINDOW_LEN * N_FEATURES}\n"
        )
        file.write(f"#define MOTOR_MODEL_N_CLASSES {N_CLASSES}\n")
        file.write(f"#define MOTOR_MODEL_N_PWM_BINS {len(PWM_BIN_NAMES)}\n")
        file.write(
            "#define MOTOR_MODEL_LOAD_PROXY_FEATURE_INDEX "
            f"{IRMS_FEATURE_INDEX}\n"
        )
        file.write(
            "#define MOTOR_MODEL_ANOMALY_THRESHOLD "
            f"{c_float(bundle['anomaly_probability_threshold'])}\n"
        )
        file.write(
            "#define MOTOR_MODEL_CLASS_CONF_THRESHOLD "
            f"{c_float(bundle['class_confidence_threshold'])}\n"
        )
        file.write(
            "#define MOTOR_MODEL_RECON_THRESHOLD "
            f"{c_float(bundle['global_reconstruction_fallback'])}\n"
        )
        # ADDED
        file.write(
            "#define MOTOR_MODEL_GLOBAL_NORMAL_BASELINE "
            f"{c_float(bundle['global_normal_baseline'])}\n"
        )
        file.write(
            "#define MOTOR_MODEL_SEVERITY_MILD_MULTIPLIER "
            f"{c_float(bundle['severity_mild_multiplier'])}\n"
        )
        file.write(
            "#define MOTOR_MODEL_SEVERITY_MODERATE_MULTIPLIER "
            f"{c_float(bundle['severity_moderate_multiplier'])}\n"
        )
        file.write(
            "#define MOTOR_MODEL_SEVERITY_SEVERE_MULTIPLIER "
            f"{c_float(bundle['severity_severe_multiplier'])}\n"
        )
        file.write(
            "#define MOTOR_MODEL_MOTOR_OFF_RECON_THRESHOLD "
            f"{c_float(bundle['state_reconstruction_thresholds']['motor_off'])}\n"
        )
        file.write(
            "#define MOTOR_MODEL_TURNING_OFF_RECON_THRESHOLD "
            f"{c_float(bundle['state_reconstruction_thresholds']['turning_off'])}\n"
        )
        file.write(
            "#define MOTOR_MODEL_TURNING_ON_RECON_THRESHOLD "
            f"{c_float(bundle['state_reconstruction_thresholds']['turning_on'])}\n\n"
        )

        file.write(
            "static const float "
            f"MOTOR_MODEL_NORMALIZATION_MEAN[{N_FEATURES}] = {{\n    "
        )
        file.write(", ".join(c_float(v) for v in deployment_mean))
        file.write("\n};\n\n")

        file.write(
            "static const float "
            f"MOTOR_MODEL_NORMALIZATION_VARIANCE[{N_FEATURES}] = {{\n    "
        )
        file.write(", ".join(c_float(v) for v in deployment_variance))
        file.write("\n};\n\n")

        file.write(
            "static const float "
            f"MOTOR_MODEL_PWM_BIN_EDGES[{len(PWM_BIN_EDGES)}] = {{\n    "
        )
        file.write(", ".join(c_float(v) for v in PWM_BIN_EDGES))
        file.write("\n};\n\n")

        file.write(
            "static const float "
            f"MOTOR_MODEL_PWM_LOAD_SPLITS[{len(load_splits)}] = {{\n    "
        )
        file.write(", ".join(c_float(v) for v in load_splits))
        file.write("\n};\n\n")

        file.write(
            "static const float "
            f"MOTOR_MODEL_NORMAL_LOW_LOAD_THRESHOLDS[{len(low_thresholds)}] = {{\n    "
        )
        file.write(", ".join(c_float(v) for v in low_thresholds))
        file.write("\n};\n\n")

        file.write(
            "static const float "
            f"MOTOR_MODEL_NORMAL_HIGH_LOAD_THRESHOLDS[{len(high_thresholds)}] = {{\n    "
        )
        file.write(", ".join(c_float(v) for v in high_thresholds))
        file.write("\n};\n\n")

        # ADDED
        file.write(
            "static const float "
            f"MOTOR_MODEL_LOCAL_NORMAL_BASELINES[{len(local_baselines)}] = {{\n    "
        )
        file.write(", ".join(c_float(v) for v in local_baselines))
        file.write("\n};\n\n")

        if quantization_metadata is not None:
            input_q = quantization_metadata["input"]
            outputs_q = quantization_metadata["outputs"]

            file.write(
                f"#define MOTOR_MODEL_INPUT_SCALE {c_float(input_q['scale'])}\n"
            )
            file.write(
                "#define MOTOR_MODEL_INPUT_ZERO_POINT "
                f"({input_q['zero_point']})\n"
            )

            for role, prefix_name in [
                ("reconstruction_output", "RECON"),
                ("anomaly_output", "ANOMALY"),
                ("class_output", "CLASS"),
            ]:
                metadata = outputs_q[role]
                file.write(
                    f"#define MOTOR_MODEL_{prefix_name}_OUTPUT_SCALE "
                    f"{c_float(metadata['scale'])}\n"
                )
                file.write(
                    f"#define MOTOR_MODEL_{prefix_name}_OUTPUT_ZERO_POINT "
                    f"({metadata['zero_point']})\n"
                )

            file.write("\n")

        file.write(
            "static const char *const "
            f"MOTOR_MODEL_LABELS[{N_CLASSES}] = {{\n"
        )
        for label in FINAL_LABELS:
            file.write(f'    "{label}",\n')
        file.write("};\n\n")
        file.write(f"#endif /* {include_guard} */\n")


TFLITE_FLOAT32_PARAMS_HEADER_PATH = os.path.join(
    FINAL_DIR, "motor_model_params_tflite_float32_safe.h",
)

write_params_header(
    TFLITE_FLOAT32_PARAMS_HEADER_PATH,
    "MOTOR_MODEL_PARAMS_TFLITE_FLOAT32_SAFE_H",
    FLOAT32_THRESHOLD_BUNDLE,
    quantization_metadata=get_tflite_quantization_metadata(
        TFLITE_FLOAT32_PATH
    ),
)

print("Saved Float32 parameters:", TFLITE_FLOAT32_PARAMS_HEADER_PATH)

In [ ]:
# ==========================================================
# CELL 26 — metadata and deployment summary
# (CHANGED: no held-out test metrics anymore — replaced with the
# 5-fold CV overall metrics and the training-set sanity check.)
# ==========================================================
METADATA_PATH = os.path.join(FINAL_DIR, "metadata.json")

metadata = {
    "model_name": "compact_conv1d_ae_anomaly_classifier_safe",
    "architecture_notes": (
        "dilation_rate=1 everywhere (no SpaceToBatchND/"
        "BatchToSpaceND), LayerNormalization instead of "
        "BatchNormalization, kernel_size=5 to recover receptive "
        "field lost by removing dilation."
    ),
    "input_shape": [WINDOW_LEN, N_FEATURES],
    "feature_columns": FEATURE_COLUMNS,
    "labels": FINAL_LABELS,
    "safe_labels": SAFE_LABELS,
    "fault_labels": FAULT_LABELS,
    "scale_divisor": SCALE_DIVISOR,
    "window_step": WINDOW_STEP,
    "row_period_sec": ROW_PERIOD_SEC,
    "parameter_count": int(final_model.count_params()),
    "normalization_mean": deployment_mean.tolist(),
    "normalization_variance": deployment_variance.tolist(),
    "final_training_epochs": FINAL_EPOCHS,
    "deployment_split": {
        "training_sessions": len(deployment_train_sessions),
        "note": (
            "Trained on 100% of sessions; validated via 5-fold CV "
            "(see cv_metrics.csv / overall_metrics.csv), not a "
            "held-out calibration/test split."
        ),
    },
    "tested_backend": "tflite_float32",
    "tflite_float32_thresholds": FLOAT32_THRESHOLD_BUNDLE,
    "tflite_float32_training_set_sanity_metrics": FLOAT32_TRAIN_METRICS,
    "cv_overall_metrics": overall_cv_metrics,
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved:", METADATA_PATH)

In [ ]:
# ==========================================================
# CELL 27 — manual test only TFLite Float32 (with full charts)
# (CHANGED: includes severity_ratio/severity_label/recommended_action
# in the output + a 5th chart plotting severity over time.)
# ==========================================================
def manual_test_float32(test_file):
    test_df = read_one(test_file)

    X_manual, starts = make_windows_from_array(
        test_df.to_numpy(dtype=np.float32)
    )

    if X_manual is None or len(X_manual) == 0:
        raise RuntimeError(
            f"The selected file has fewer than {WINDOW_LEN} valid rows."
        )

    predictions = predict_tflite(TFLITE_FLOAT32_PATH, X_manual)

    (
        reconstruction_mse, mean_pwm, mean_irms,
    ) = calculate_backend_values(X_manual, predictions)

    reconstruction_thresholds = thresholds_for_windows(
        FLOAT32_THRESHOLD_BUNDLE, predictions["class_output"],
        mean_pwm, mean_irms,
    )

    local_baselines = local_baselines_for_windows(
        FLOAT32_THRESHOLD_BUNDLE, mean_pwm,
    )

    decisions = fuse_backend_decisions(
        class_probabilities=predictions["class_output"],
        anomaly_probabilities=predictions["anomaly_output"],
        reconstruction_mse=reconstruction_mse,
        reconstruction_thresholds=reconstruction_thresholds,
        local_baselines=local_baselines,
        threshold_bundle=FLOAT32_THRESHOLD_BUNDLE,
    )

    time_sec = (starts + WINDOW_LEN / 2.0) * ROW_PERIOD_SEC

    result = pd.DataFrame({
        "window": np.arange(len(X_manual), dtype=np.int64),
        "start_row": starts,
        "time_sec": time_sec,
        "mean_pwm": mean_pwm,
        "mean_irms": mean_irms,
        "pred_class_id": decisions["class_ids"],
        "pred_label": [
            FINAL_LABELS[i] for i in decisions["class_ids"]
        ],
        "class_confidence": decisions["class_confidence"],
        "anomaly_probability": predictions["anomaly_output"],
        "reconstruction_mse": reconstruction_mse,
        "reconstruction_threshold": reconstruction_thresholds,
        "local_baseline": local_baselines,
        "severity_ratio": decisions["severity_ratio"],
        "severity_label": decisions["severity_label"],
        "recommended_action": decisions["recommended_action"],
        "final_anomaly_flag":
            decisions["final_anomaly_flags"].astype(np.int64),
        "final_decision": decisions["final_labels"],
    })

    for class_id, label in enumerate(FINAL_LABELS):
        result[f"probability_{label}"] = (
            predictions["class_output"][:, class_id]
        )

    stem = Path(test_file).stem

    output_csv = os.path.join(
        MANUAL_TEST_DIR, f"manual_{stem}_tflite_float32_safe.csv",
    )
    result.to_csv(output_csv, index=False)

    majority_class = result["pred_label"].value_counts().idxmax()
    majority_decision = result["final_decision"].value_counts().idxmax()
    majority_severity = result["severity_label"].value_counts().idxmax()

    print("\n" + "=" * 70)
    print("MANUAL TEST — TFLITE FLOAT32 (STM32-SAFE)")
    print("=" * 70)
    print("Test file:", test_file)
    print("Total windows:", len(result))
    print("Majority predicted class:", majority_class)
    print("Majority final decision:", majority_decision)
    print("Majority severity tier:", majority_severity)
    print(
        "Mean class confidence:",
        round(float(result["class_confidence"].mean()), 6),
    )
    print(
        "Mean anomaly probability:",
        round(float(result["anomaly_probability"].mean()), 6),
    )
    print(
        "Anomalous windows:",
        int(result["final_anomaly_flag"].sum()), "/", len(result),
    )
    print("Saved CSV:", output_csv)

    print("\nFinal decisions:")
    print(result["final_decision"].value_counts())

    print("\nSeverity tiers:")
    print(result["severity_label"].value_counts())

    # ======================================================
    # CHART 1 — multi-class classification head
    # ======================================================
    plt.figure(figsize=(14, 7))

    for label in FINAL_LABELS:
        plt.plot(
            result["time_sec"],
            result[f"probability_{label}"],
            label=label,
            linewidth=1.5,
        )

    plt.axhline(
        y=FLOAT32_THRESHOLD_BUNDLE["class_confidence_threshold"],
        linestyle="--",
        label=(
            f"Confidence threshold "
            f"({FLOAT32_THRESHOLD_BUNDLE['class_confidence_threshold']:.2f})"
        ),
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Class probability")
    plt.ylim(0.0, 1.05)
    plt.title(f"Classification head probabilities — {stem}")
    plt.legend(loc="upper right", ncol=2)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    class_head_path = os.path.join(
        MANUAL_TEST_DIR, f"manual_test_{stem}_class_head.png",
    )
    plt.savefig(class_head_path, dpi=180)
    plt.show()
    plt.close()

    print("Saved classification-head chart:", class_head_path)

    # ======================================================
    # CHART 2 — binary anomaly head
    # ======================================================
    anomaly_prob_threshold = FLOAT32_THRESHOLD_BUNDLE.get(
        "anomaly_probability_threshold", 0.5
    )

    plt.figure(figsize=(14, 5))

    plt.plot(
        result["time_sec"],
        result["anomaly_probability"],
        label="Anomaly probability",
        linewidth=2,
    )

    plt.axhline(
        y=anomaly_prob_threshold,
        linestyle="--",
        label=f"Anomaly threshold ({anomaly_prob_threshold:.2f})",
    )

    plt.fill_between(
        result["time_sec"],
        0,
        1,
        where=(result["anomaly_probability"] >= anomaly_prob_threshold),
        alpha=0.15,
        label="Anomaly detected",
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Anomaly probability")
    plt.ylim(0.0, 1.05)
    plt.title(f"Binary anomaly head — {stem}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    anomaly_head_path = os.path.join(
        MANUAL_TEST_DIR, f"manual_test_{stem}_anomaly_head.png",
    )
    plt.savefig(anomaly_head_path, dpi=180)
    plt.show()
    plt.close()

    print("Saved anomaly-head chart:", anomaly_head_path)

    # ======================================================
    # CHART 3 — reconstruction autoencoder head
    # ======================================================
    plt.figure(figsize=(14, 5))

    plt.plot(
        result["time_sec"],
        result["reconstruction_mse"],
        linewidth=2,
        label="Reconstruction MSE",
        color="blue",
    )

    plt.plot(
        result["time_sec"],
        result["reconstruction_threshold"],
        linestyle="--",
        linewidth=2,
        label="State/PWM/load-conditioned threshold",
        color="red",
    )

    plt.fill_between(
        result["time_sec"],
        0,
        result["reconstruction_mse"],
        where=(
            result["reconstruction_mse"]
            > result["reconstruction_threshold"]
        ),
        alpha=0.15,
        label="Reconstruction anomaly",
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Reconstruction MSE")
    plt.title(f"Autoencoder reconstruction head — {stem}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    reconstruction_head_path = os.path.join(
        MANUAL_TEST_DIR, f"manual_test_{stem}_reconstruction_head.png",
    )
    plt.savefig(reconstruction_head_path, dpi=180)
    plt.show()
    plt.close()

    print("Saved reconstruction-head chart:", reconstruction_head_path)

    # ======================================================
    # CHART 4 — predicted class timeline
    # ======================================================
    plt.figure(figsize=(14, 5))

    plt.step(
        result["time_sec"],
        result["pred_class_id"],
        where="mid",
        linewidth=2,
    )

    plt.yticks(np.arange(N_CLASSES), FINAL_LABELS)
    plt.xlabel("Time [s]")
    plt.ylabel("Predicted class")
    plt.title(f"Classification result timeline — {stem}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    timeline_path = os.path.join(
        MANUAL_TEST_DIR, f"manual_test_{stem}_class_timeline.png",
    )
    plt.savefig(timeline_path, dpi=180)
    plt.show()
    plt.close()

    print("Saved class timeline:", timeline_path)

    # ======================================================
    # CHART 5 (ADDED) — severity ratio over time
    # ======================================================
    plt.figure(figsize=(14, 5))

    plt.plot(
        result["time_sec"],
        result["severity_ratio"],
        linewidth=2,
        color="darkorange",
        label="Severity ratio (MSE / local baseline)",
    )

    for level, style, color in [
        (SEVERITY_MILD_MULTIPLIER, "--", "gold"),
        (SEVERITY_MODERATE_MULTIPLIER, "--", "orange"),
        (SEVERITY_SEVERE_MULTIPLIER, "--", "red"),
    ]:
        plt.axhline(y=level, linestyle=style, color=color, alpha=0.7)

    plt.xlabel("Time [s]")
    plt.ylabel("Severity ratio")
    plt.title(f"Reconstruction severity — {stem}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    severity_path = os.path.join(
        MANUAL_TEST_DIR, f"manual_test_{stem}_severity.png",
    )
    plt.savefig(severity_path, dpi=180)
    plt.show()
    plt.close()

    print("Saved severity chart:", severity_path)

    return result

In [ ]:
# ==========================================================
# CELL 28 — select manual test file
# ==========================================================
try:
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    TEST_FILE = filedialog.askopenfilename(
        title="Select motor CSV or Excel file",
        filetypes=[
            ("Excel files", "*.xlsx"),
            ("CSV files", "*.csv"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if TEST_FILE:
        manual_result = manual_test_float32(TEST_FILE)
    else:
        print("No manual test file selected.")
except Exception as error:
    print("Manual file dialog unavailable:", error)

In [ ]:
# ==========================================================
# CELL 29 — STANDALONE CNN MANUAL TEST
# Run ONLY this cell — no previous cells required.
# Uses saved CNN Float32 TFLite model and thresholds.
# Preserves original CNN backend decision fusion.
# ==========================================================

import json
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


# ==========================================================
# 1. LOCATE SAVED CNN MODEL
# ==========================================================

OUTPUT_ROOT = Path.cwd() / "conv1d_ae_cnn_3head_safe_output"
MODEL_DIR = OUTPUT_ROOT / "final_model"

MODEL_PATH = MODEL_DIR / (
    "compact_conv1d_ae_anomaly_classifier_safe_float32.tflite"
)

METADATA_PATH = MODEL_DIR / "metadata.json"
THRESHOLDS_PATH = MODEL_DIR / "thresholds_tflite_float32.json"


def select_file(title, filetypes, initialdir=None):
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    path = filedialog.askopenfilename(
        title=title,
        initialdir=str(initialdir or Path.cwd()),
        filetypes=filetypes
    )
    root.destroy()
    return Path(path) if path else None


if not MODEL_PATH.is_file():
    MODEL_PATH = select_file(
        "Select trained CNN Float32 TFLite model",
        [("TFLite model", "*.tflite")]
    )
    if MODEL_PATH is None:
        raise RuntimeError("No CNN model selected.")

    MODEL_DIR = MODEL_PATH.parent
    OUTPUT_ROOT = MODEL_DIR.parent
    METADATA_PATH = MODEL_DIR / "metadata.json"
    THRESHOLDS_PATH = MODEL_DIR / "thresholds_tflite_float32.json"

if not METADATA_PATH.is_file():
    raise FileNotFoundError(METADATA_PATH)

with open(METADATA_PATH, encoding="utf-8") as f:
    meta = json.load(f)

if THRESHOLDS_PATH.is_file():
    with open(THRESHOLDS_PATH, encoding="utf-8") as f:
        bundle = json.load(f)
else:
    bundle = meta["tflite_float32_thresholds"]

FEATURES = meta["feature_columns"]
LABELS = meta["labels"]
SAFE_LABELS = meta["safe_labels"]
FAULT_LABELS = meta["fault_labels"]

WIN, NF = map(int, meta["input_shape"])
STEP = int(meta["window_step"])
ROW_SEC = float(meta["row_period_sec"])
SCALE = float(meta["scale_divisor"])

MEAN = np.asarray(meta["normalization_mean"], np.float32)
VAR = np.asarray(meta["normalization_variance"], np.float32)

PWM_NAMES = bundle["pwm_bin_names"]
PWM_EDGES = np.asarray(bundle["pwm_bin_edges"], np.float32)

ANOM_THR = float(bundle["anomaly_probability_threshold"])
CLASS_THR = float(bundle["class_confidence_threshold"])
GLOBAL_RECON_THR = float(bundle["global_reconstruction_fallback"])
GLOBAL_BASELINE = float(bundle["global_normal_baseline"])

MILD = float(bundle["severity_mild_multiplier"])
MODERATE = float(bundle["severity_moderate_multiplier"])
SEVERE = float(bundle["severity_severe_multiplier"])

BATCH_SIZE = 128

print("CNN Float32 model:", MODEL_PATH)
print("Input:", WIN, "x", NF)


# ==========================================================
# 2. SELECT MANUAL TEST FILE
# ==========================================================

TEST_FILE = select_file(
    "Select CNN manual motor test file",
    [
        ("Motor data", "*.xlsx *.csv"),
        ("Excel", "*.xlsx"),
        ("CSV", "*.csv"),
        ("All files", "*.*")
    ]
)

if TEST_FILE is None:
    raise RuntimeError("Manual test cancelled.")

RESULT_DIR = OUTPUT_ROOT / "manual_test_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
stem = TEST_FILE.stem


# ==========================================================
# 3. READ AND PREPROCESS TEST FILE
# ==========================================================

if TEST_FILE.suffix.lower() == ".xlsx":
    df = pd.read_excel(TEST_FILE)
elif TEST_FILE.suffix.lower() == ".csv":
    df = pd.read_csv(TEST_FILE)
else:
    raise ValueError("Only CSV and XLSX supported.")

missing = [c for c in FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

data = df[FEATURES].apply(pd.to_numeric, errors="coerce")
data = data / SCALE
data["PWM"] = data["PWM"].clip(upper=1.0)
data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna().reset_index(drop=True)

X = data.to_numpy(np.float32)

if len(X) < WIN:
    raise ValueError(f"At least {WIN} valid rows required.")

starts = np.arange(0, len(X) - WIN + 1, STEP)
windows = np.stack([
    X[s:s + WIN] for s in starts
]).astype(np.float32)

print("Test file:", TEST_FILE.name)
print("Valid rows:", len(X))
print("Windows:", len(windows))


# ==========================================================
# 4. CNN FLOAT32 TFLITE INFERENCE
# ==========================================================

interpreter = tf.lite.Interpreter(model_path=str(MODEL_PATH))
interpreter.allocate_tensors()

input_detail = interpreter.get_input_details()[0]

if input_detail["dtype"] != np.float32:
    raise TypeError(
        "Selected model is not Float32 TFLite. "
        "Choose the *_float32.tflite model."
    )

outputs = {
    "reconstruction_output": [],
    "anomaly_output": [],
    "class_output": []
}

for start in range(0, len(windows), BATCH_SIZE):
    batch = windows[start:start + BATCH_SIZE]
    required_shape = [len(batch), WIN, NF]

    if list(input_detail["shape"]) != required_shape:
        interpreter.resize_tensor_input(
            input_detail["index"], required_shape, strict=False
        )
        interpreter.allocate_tensors()
        input_detail = interpreter.get_input_details()[0]

    interpreter.set_tensor(input_detail["index"], batch)
    interpreter.invoke()

    for detail in interpreter.get_output_details():
        values = interpreter.get_tensor(detail["index"])
        elements = int(np.prod(values.shape[1:]))

        if elements == WIN * NF:
            role = "reconstruction_output"
            values = values.reshape(-1, WIN, NF)
        elif elements == len(LABELS):
            role = "class_output"
            values = values.reshape(-1, len(LABELS))
        elif elements == 1:
            role = "anomaly_output"
            values = values.reshape(-1)
        else:
            raise RuntimeError(
                f"Unexpected CNN output: {detail['name']}"
            )

        outputs[role].append(values.astype(np.float32))

predictions = {
    key: np.concatenate(value, axis=0)
    for key, value in outputs.items()
}

reconstructed = predictions["reconstruction_output"]
anomaly_probability = predictions["anomaly_output"]
class_probability = predictions["class_output"]


# ==========================================================
# 5. RECONSTRUCTION MSE
# ==========================================================

normalized = (
    (windows - MEAN.reshape(1, 1, NF))
    / np.sqrt(VAR.reshape(1, 1, NF) + 1e-7)
)

reconstruction_mse = np.mean(
    np.square(reconstructed - normalized), axis=(1, 2)
).astype(np.float32)

mean_pwm = windows[:, :, FEATURES.index("PWM")].mean(axis=1)
mean_irms = windows[
    :, :, FEATURES.index(bundle["load_proxy_feature"])
].mean(axis=1)


# ==========================================================
# 6. CNN STATE/PWM/LOAD-CONDITIONED THRESHOLDS
# ==========================================================

def pwm_bin_index(pwm):
    return int(np.clip(
        np.searchsorted(PWM_EDGES, pwm, side="right") - 1,
        0, len(PWM_NAMES) - 1
    ))


def get_recon_threshold(cid, pwm, irms):
    label = LABELS[int(cid)]

    if label in ["motor_off", "turning_off", "turning_on"]:
        return float(
            bundle["state_reconstruction_thresholds"].get(
                label, GLOBAL_RECON_THR
            )
        )

    name = PWM_NAMES[pwm_bin_index(pwm)]
    split = float(bundle["pwm_load_splits"].get(
        name, bundle["global_normal_load_split"]
    ))
    state = "high_load" if irms >= split else "low_load"

    return float(
        bundle["normal_pwm_load_reconstruction_thresholds"]
        .get(name, {})
        .get(state, bundle["normal_pwm_reconstruction_thresholds"]
             .get(name, GLOBAL_RECON_THR))
    )


class_id = np.argmax(class_probability, axis=1).astype(np.int64)
class_confidence = np.max(class_probability, axis=1)

sorted_probs = np.sort(class_probability, axis=1)
class_margin = sorted_probs[:, -1] - sorted_probs[:, -2]

classifier_reliable = (
    (class_confidence >= CLASS_THR)
    | (class_margin >= 0.25)
)

conditional_thresholds = np.asarray([
    get_recon_threshold(cid, pwm, irms)
    for cid, pwm, irms in zip(class_id, mean_pwm, mean_irms)
], dtype=np.float32)

# Original CNN Cell 22: global fallback for unreliable classes.
recon_thresholds = np.where(
    classifier_reliable,
    conditional_thresholds,
    GLOBAL_RECON_THR
).astype(np.float32)

local_baselines = np.asarray([
    float(bundle["normal_pwm_local_baselines"].get(
        PWM_NAMES[pwm_bin_index(pwm)], GLOBAL_BASELINE
    ))
    for pwm in mean_pwm
], dtype=np.float32)


# ==========================================================
# 7. ORIGINAL CNN THREE-HEAD DECISION FUSION
# ==========================================================

binary_flags = anomaly_probability >= ANOM_THR
reconstruction_flags = reconstruction_mse > recon_thresholds
evidence_flags = binary_flags | reconstruction_flags

fault_class_flags = (
    np.isin(class_id, [
        LABELS.index(label) for label in FAULT_LABELS
    ]) & classifier_reliable
)

safe_class_flags = (
    np.isin(class_id, [
        LABELS.index(label) for label in SAFE_LABELS
    ]) & classifier_reliable
)

anomaly_flags = evidence_flags | fault_class_flags

final_decisions = []

for i, cid in enumerate(class_id):
    label = LABELS[int(cid)]

    if evidence_flags[i]:
        if fault_class_flags[i]:
            decision = label
        else:
            decision = "unknown_fault"
    elif safe_class_flags[i]:
        decision = label
    elif fault_class_flags[i]:
        decision = "unknown_fault"
    else:
        decision = "uncertain_safe"

    final_decisions.append(decision)


# ==========================================================
# 8. RECONSTRUCTION SEVERITY
# ==========================================================

safe_baselines = np.where(
    local_baselines > 0, local_baselines, 1.0
)

severity_ratio = reconstruction_mse / safe_baselines
severity_labels = []
recommended_actions = []

for ratio in severity_ratio:
    if ratio < MILD:
        severity_labels.append("normal")
        recommended_actions.append("No action needed.")
    elif ratio < MODERATE:
        severity_labels.append("mild")
        recommended_actions.append(
            "Log and monitor; no immediate action."
        )
    elif ratio < SEVERE:
        severity_labels.append("moderate")
        recommended_actions.append("Schedule inspection soon.")
    else:
        severity_labels.append("severe")
        recommended_actions.append(
            "Stop motor / inspect immediately."
        )


# ==========================================================
# 9. EARLY WARNING — ADDITIONAL INFORMATION
# ==========================================================

early_candidates = (
    (class_id == LABELS.index("normal"))
    & reconstruction_flags
)

early_streak = np.zeros(len(windows), dtype=np.int32)

for i, candidate in enumerate(early_candidates):
    if candidate:
        early_streak[i] = (
            early_streak[i - 1] + 1 if i > 0 else 1
        )

early_flags = early_streak >= 3


# ==========================================================
# 10. SAVE DETAILED MANUAL TEST CSV
# ==========================================================

time_sec = (starts + WIN / 2.0) * ROW_SEC

result = pd.DataFrame({
    "window": np.arange(len(windows)),
    "start_row": starts,
    "time_sec": time_sec,
    "mean_pwm": mean_pwm,
    "mean_irms": mean_irms,
    "pred_class_id": class_id,
    "pred_label": [LABELS[int(i)] for i in class_id],
    "class_confidence": class_confidence,
    "class_margin": class_margin,
    "classifier_reliable": classifier_reliable.astype(int),
    "anomaly_probability": anomaly_probability,
    "binary_anomaly_flag": binary_flags.astype(int),
    "reconstruction_mse": reconstruction_mse,
    "reconstruction_threshold": recon_thresholds,
    "local_baseline": local_baselines,
    "severity_ratio": severity_ratio,
    "severity_label": severity_labels,
    "recommended_action": recommended_actions,
    "reconstruction_anomaly_flag": reconstruction_flags.astype(int),
    "final_anomaly_flag": anomaly_flags.astype(int),
    "final_decision": final_decisions,
    "early_warning_candidate": early_candidates.astype(int),
    "early_warning_streak": early_streak,
    "early_warning_flag": early_flags.astype(int)
})

for i, label in enumerate(LABELS):
    result[f"probability_{label}"] = class_probability[:, i]

CSV_PATH = RESULT_DIR / f"manual_{stem}_tflite_float32_safe.csv"
result.to_csv(CSV_PATH, index=False)


# ==========================================================
# 11. PRINT CNN MANUAL TEST SUMMARY
# ==========================================================

print("\n" + "=" * 70)
print("MANUAL CNN 3-HEAD TEST — TFLITE FLOAT32")
print("=" * 70)
print("File:", TEST_FILE.name)
print("Total windows:", len(result))
print("Majority class:", result["pred_label"].value_counts().idxmax())
print("Majority decision:", result["final_decision"].value_counts().idxmax())
print("Majority severity:", result["severity_label"].value_counts().idxmax())
print("Mean class confidence:", float(class_confidence.mean()))
print("Mean anomaly probability:", float(anomaly_probability.mean()))
print("Mean reconstruction MSE:", float(reconstruction_mse.mean()))
print("Anomalous windows:", int(anomaly_flags.sum()), "/", len(result))
print("Early warnings:", int(early_flags.sum()), "/", len(result))

if early_flags.any():
    print("First early warning:", float(
        time_sec[np.flatnonzero(early_flags)[0]]
    ), "s")

print("\nFinal decisions:")
print(result["final_decision"].value_counts())
print("\nSeverity tiers:")
print(result["severity_label"].value_counts())
print("\nSaved CSV:", CSV_PATH)


# ==========================================================
# 12. SAVE ORIGINAL FIVE CNN GRAPHS
# ==========================================================

def save_chart(suffix):
    path = RESULT_DIR / f"manual_test_{stem}_{suffix}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.show()
    plt.close()
    print("Saved:", path)


# CHART 1 — CNN classification probabilities
plt.figure(figsize=(14, 7))

for i, label in enumerate(LABELS):
    plt.plot(time_sec, class_probability[:, i],
             label=label, linewidth=1.5)

plt.axhline(CLASS_THR, linestyle="--",
            label=f"Confidence threshold ({CLASS_THR:.2f})")
plt.xlabel("Time [s]")
plt.ylabel("Class probability")
plt.ylim(0, 1.05)
plt.title(f"CNN classification head probabilities — {stem}")
plt.legend(loc="upper right", ncol=2)
plt.grid(True, alpha=0.3)
save_chart("class_head")


# CHART 2 — CNN binary anomaly head
plt.figure(figsize=(14, 5))

plt.plot(time_sec, anomaly_probability,
         label="Anomaly probability", linewidth=2)
plt.axhline(ANOM_THR, linestyle="--",
            label=f"Anomaly threshold ({ANOM_THR:.2f})")
plt.fill_between(time_sec, 0, 1, where=binary_flags,
                 alpha=0.15, label="Anomaly detected")
plt.xlabel("Time [s]")
plt.ylabel("Anomaly probability")
plt.ylim(0, 1.05)
plt.title(f"CNN binary anomaly head — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)
save_chart("anomaly_head")


# CHART 3 — CNN reconstruction MSE
plt.figure(figsize=(14, 5))

plt.plot(time_sec, reconstruction_mse,
         linewidth=2, label="Reconstruction MSE")
plt.plot(time_sec, recon_thresholds,
         linestyle="--", linewidth=2,
         label="Effective state/PWM/load threshold")
plt.fill_between(time_sec, 0, reconstruction_mse,
                 where=reconstruction_flags, alpha=0.15,
                 label="Reconstruction anomaly")
plt.xlabel("Time [s]")
plt.ylabel("Reconstruction MSE")
plt.title(f"CNN autoencoder reconstruction — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)
save_chart("reconstruction_head")


# CHART 4 — CNN predicted class timeline
plt.figure(figsize=(14, 5))

plt.step(time_sec, class_id, where="mid", linewidth=2)
plt.yticks(np.arange(len(LABELS)), LABELS)
plt.xlabel("Time [s]")
plt.ylabel("Predicted class")
plt.title(f"CNN class timeline — {stem}")
plt.grid(True, alpha=0.3)
save_chart("class_timeline")


# CHART 5 — CNN severity ratio
plt.figure(figsize=(14, 5))

plt.plot(time_sec, severity_ratio, linewidth=2,
         color="darkorange",
         label="Severity ratio (MSE / local baseline)")

for level, color in [
    (MILD, "gold"),
    (MODERATE, "orange"),
    (SEVERE, "red")
]:
    plt.axhline(level, linestyle="--",
                color=color, alpha=0.7)

plt.xlabel("Time [s]")
plt.ylabel("Severity ratio")
plt.title(f"CNN reconstruction severity — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)
save_chart("severity")


# ==========================================================
# 13. COMPLETE
# ==========================================================

manual_result = result

print("\n" + "=" * 70)
print("CNN MANUAL TEST COMPLETED")
print("=" * 70)
print("Model:", MODEL_PATH.name)
print("Test file:", TEST_FILE.name)
print("Result directory:", RESULT_DIR)
print("CSV and five charts saved.")
print("No training or recalibration performed.")